In [1]:
!pip install --upgrade pip setuptools wheel -q
!pip install --upgrade cmake -q
!pip install scs --prefer-binary -q
!pip install cvxpy --prefer-binary -q
!pip install awswrangler -q
!pip install optbinning -q
!pip install lightgbm
!pip install xgboost
!pip install xgboost --prefer-binary
!pip install catboost


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip
ERROR: To modify pip, please run the following command:
C:\Program Files\Python314\python.exe -m pip install --upgrade pip setuptools wheel -q

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program F

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Program Files\Python314\python.exe -m pip install --upgrade pip


In [2]:
import awswrangler as wr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from optbinning import BinningProcess
import shutil
from warnings import simplefilter
simplefilter(action = "ignore") #, category = FutureWarning

pd.set_option('display.max_rows', 500)
from sklearn.preprocessing import LabelEncoder

(CVXPY) Jul 28 09:36:04 AM: Encountered unexpected exception importing solver GLOP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')
(CVXPY) Jul 28 09:36:04 AM: Encountered unexpected exception importing solver PDLP:
RuntimeError('Unrecognized new version of ortools (9.15.6755). Expected < 9.15.0. Please open a feature request on cvxpy to enable support for this version.')


In [3]:
import pandas as pd
import numpy as np
import seaborn as sb
import matplotlib.pyplot as plt
from typing import List, Tuple
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os
import plotly.express as px

pd.set_option('display.float_format', '{:.2f}'.format)

ModuleNotFoundError: No module named 'seaborn'

In [4]:
# === Conexion Athena estilo Cruce + fallback awswrangler ===
import os
import re
import sys
from pathlib import Path

import boto3
import awswrangler as wr
import pandas as pd

EXPLICIT_CREDENTIALS_SH = Path(r"c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test/credentials.sh")


def _find_dir_with_athena_client(preferred_dir: Path | None = None) -> Path | None:
    cwd = Path.cwd().resolve()
    search_roots = []

    if preferred_dir is not None:
        search_roots.append(preferred_dir)

    search_roots.extend([cwd, *cwd.parents])

    home = Path.home()
    search_roots.extend([
        home / "OneDrive - Interbank" / "conexion_aws" / "athena_conection_test",
        Path("c:/Users/b46637/OneDrive - Interbank/conexion_aws/athena_conection_test"),
    ])

    visited = set()
    for root in search_roots:
        if root in visited:
            continue
        visited.add(root)

        if not root.exists():
            continue
        if (root / "athena_client.py").exists() and (root / "athena_config.json").exists():
            return root
    return None


def _load_credentials_from_sh(sh_path: Path) -> list[str]:
    if not sh_path.exists():
        return []

    loaded_keys: list[str] = []
    pattern = re.compile(r'^\s*export\s+([A-Za-z_][A-Za-z0-9_]*)=(.*)$')

    for line in sh_path.read_text(encoding="utf-8", errors="ignore").splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        match = pattern.match(line)
        if not match:
            continue

        key, raw_val = match.groups()
        value = raw_val.strip().strip('"').strip("'")
        if key and value:
            os.environ[key] = value
            loaded_keys.append(key)

    return loaded_keys


def _build_session(aws_region: str) -> boto3.Session:
    aws_profile = os.getenv("AWS_PROFILE")
    if aws_profile:
        return boto3.Session(profile_name=aws_profile, region_name=aws_region)
    return boto3.Session(region_name=aws_region)


def _session_is_valid(sess: boto3.Session) -> tuple[bool, str | None]:
    try:
        sts = sess.client("sts")
        _ = sts.get_caller_identity()
        return True, None
    except Exception as exc:
        return False, str(exc)


ATHENA_MODE = "wrangler"
ATHENA_DATABASE = os.getenv("ATHENA_DATABASE", "disc_comercial")
ATHENA_WORKGROUP = os.getenv("ATHENA_WORKGROUP", "primary")
ATHENA_OUTPUT = os.getenv(
    "ATHENA_OUTPUT",
    "s3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/athena_results/"
 )
AWS_REGION = os.getenv("AWS_REGION", "us-east-1")

client = None

credentials_file = EXPLICIT_CREDENTIALS_SH if EXPLICIT_CREDENTIALS_SH.exists() else None
if credentials_file is None:
    print(f"⚠ No se encontró credentials.sh en ruta fija: {EXPLICIT_CREDENTIALS_SH}")

preferred_dir = credentials_file.parent if credentials_file is not None else None
athena_dir = _find_dir_with_athena_client(preferred_dir=preferred_dir)
loaded_cred_keys: list[str] = []

if credentials_file is not None:
    loaded_cred_keys = _load_credentials_from_sh(credentials_file)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {credentials_file}")
elif athena_dir is not None:
    fallback_sh = athena_dir / "credentials.sh"
    loaded_cred_keys = _load_credentials_from_sh(fallback_sh)
    if loaded_cred_keys:
        print(f"✓ Credenciales cargadas desde: {fallback_sh}")

try:
    session = _build_session(AWS_REGION)
except Exception:
    session = boto3.Session(region_name=AWS_REGION)

ok_session, session_error = _session_is_valid(session)
if not ok_session and session_error and "ExpiredToken" in session_error and loaded_cred_keys:
    print("⚠ Se detectó token expirado en credentials.sh. Reintentando con credenciales locales (perfil/default)...")
    for key in ["AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "AWS_SESSION_TOKEN"]:
        os.environ.pop(key, None)
    session = _build_session(AWS_REGION)

if athena_dir is not None:
    if str(athena_dir) not in sys.path:
        sys.path.append(str(athena_dir))
    try:
        from athena_client import AthenaClient

        if credentials_file is None:
            credentials_file = athena_dir / "credentials.sh"

        client = AthenaClient(
            credentials_file=str(credentials_file),
            config_file=str(athena_dir / "athena_config.json"),
        )
        ATHENA_MODE = "athena_client"
        print(f"✓ AthenaClient cargado desde: {athena_dir}")
    except Exception as exc:
        print(f"⚠ No se pudo inicializar AthenaClient ({exc}). Se usará awswrangler.")
else:
    print("⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.")


def athena_query(query: str, database: str = ATHENA_DATABASE) -> pd.DataFrame:
    if ATHENA_MODE == "athena_client" and client is not None:
        return client.query(query)
    return wr.athena.read_sql_query(
        sql=query,
        database=database,
        ctas_approach=False,
        boto3_session=session,
        workgroup=ATHENA_WORKGROUP,
        s3_output=ATHENA_OUTPUT,
    )


def s3_read_csv(path: str, sep: str = "|", **kwargs) -> pd.DataFrame:
    return wr.s3.read_csv(path=path, sep=sep, boto3_session=session, **kwargs)


def test_aws_connection(sample_s3_path: str | None = None) -> None:
    sts = session.client("sts")
    ident = sts.get_caller_identity()
    print(f"✓ AWS Account: {ident.get('Account')} | ARN: {ident.get('Arn')}")

    if sample_s3_path:
        _ = wr.s3.read_csv(path=sample_s3_path, sep='|', boto3_session=session, nrows=1)
        print(f"✓ Lectura S3 OK: {sample_s3_path}")


print(f"Modo Athena activo: {ATHENA_MODE}")
print(f"DB: {ATHENA_DATABASE} | WG: {ATHENA_WORKGROUP}")
print(f"credentials.sh en uso: {credentials_file}")
print("Helper Athena: athena_query(query)")
print("Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')")
print("Diagnóstico opcional: test_aws_connection()")

✓ Credenciales cargadas desde: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
⚠ No se encontró athena_client.py + athena_config.json. Se usará awswrangler.
Modo Athena activo: wrangler
DB: disc_comercial | WG: primary
credentials.sh en uso: c:\Users\b46637\OneDrive - Interbank\conexion_aws\athena_conection_test\credentials.sh
Helper Athena: athena_query(query)
Helper S3 CSV: s3_read_csv('s3://bucket/prefix/file.txt', sep='|')
Diagnóstico opcional: test_aws_connection()


In [ ]:
%%time
query = """SELECT
    -- Claves y metadatos básicos
    DISTINCT(a.num_documento) AS num_documento,
    CAST(a.mes_base AS INTEGER) AS mes_base,
    a.codunico,
    a.tipo_documento,
    a.tipo_cliente,
    a.subsegmento,
    
    -- Fechas
    TRY_CAST(a.fecha_vinculacion AS DATE) AS fecha_vinculacion,
    TRY_CAST(a.fecha_nacimiento AS DATE) AS fecha_nacimiento,
    TRY_CAST(a.fecha_mas_antiguo_lsb AS DATE) AS fecha_mas_antiguo_lsb,
    TRY_CAST(a.fecha_mas_reciente_lsb AS DATE) AS fecha_mas_reciente_lsb,
    TRY_CAST(a.fecha_ejecucion AS DATE) AS fecha_ejecucion,
    TRY_CAST(a.ultima_fecha_r1 AS DATE) AS ultima_fecha_r1,
    TRY_CAST(a.fecha_ultimo_kyc_hist AS TIMESTAMP) AS fecha_ultimo_kyc_hist,
    TRY_CAST(a.fecha_constitucion AS DATE) AS fecha_constitucion,
    TRY_CAST(a.fecha_ultimo_caso AS TIMESTAMP) AS fecha_ultimo_caso,
    TRY_CAST(a.fecha_ultima_alerta AS TIMESTAMP) AS fecha_ultima_alerta,
    TRY_CAST(a.fecha_ultimo_ros AS TIMESTAMP) AS fecha_ultimo_ros,

    -- Numéricos / decimales
    TRY_CAST(a.pasivo_soles AS DOUBLE) AS pasivo_soles,
    TRY_CAST(a.trx_monto_abonos_1m_total AS DOUBLE) AS trx_monto_abonos_1m_total,
    TRY_CAST(a.trx_monto_abonos_3m_total AS DOUBLE) AS trx_monto_abonos_3m_total,
    TRY_CAST(a.trx_monto_abonos_6m_total AS DOUBLE) AS trx_monto_abonos_6m_total,
    TRY_CAST(a.trx_monto_abonos_9m_total AS DOUBLE) AS trx_monto_abonos_9m_total,
    TRY_CAST(a.trx_monto_abonos_12m_total AS DOUBLE) AS trx_monto_abonos_12m_total,
    TRY_CAST(a.trx_monto_cargos_1m_total AS DOUBLE) AS trx_monto_cargos_1m_total,
    TRY_CAST(a.trx_monto_cargos_3m_total AS DOUBLE) AS trx_monto_cargos_3m_total,
    TRY_CAST(a.trx_monto_cargos_6m_total AS DOUBLE) AS trx_monto_cargos_6m_total,
    TRY_CAST(a.trx_monto_cargos_9m_total AS DOUBLE) AS trx_monto_cargos_9m_total,
    TRY_CAST(a.trx_monto_cargos_12m_total AS DOUBLE) AS trx_monto_cargos_12m_total,
    TRY_CAST(a.trx_monto_abonos_1m_efectivo AS DOUBLE) AS trx_monto_abonos_1m_efectivo,
    TRY_CAST(a.trx_monto_abonos_3m_efectivo AS DOUBLE) AS trx_monto_abonos_3m_efectivo,
    TRY_CAST(a.trx_monto_abonos_6m_efectivo AS DOUBLE) AS trx_monto_abonos_6m_efectivo,
    TRY_CAST(a.trx_monto_abonos_9m_efectivo AS DOUBLE) AS trx_monto_abonos_9m_efectivo,
    TRY_CAST(a.trx_monto_abonos_12m_efectivo AS DOUBLE) AS trx_monto_abonos_12m_efectivo,
    TRY_CAST(a.trx_monto_cargos_1m_efectivo AS DOUBLE) AS trx_monto_cargos_1m_efectivo,
    TRY_CAST(a.trx_monto_cargos_3m_efectivo AS DOUBLE) AS trx_monto_cargos_3m_efectivo,
    TRY_CAST(a.trx_monto_cargos_6m_efectivo AS DOUBLE) AS trx_monto_cargos_6m_efectivo,
    TRY_CAST(a.trx_monto_cargos_9m_efectivo AS DOUBLE) AS trx_monto_cargos_9m_efectivo,
    TRY_CAST(a.trx_monto_cargos_12m_efectivo AS DOUBLE) AS trx_monto_cargos_12m_efectivo,
    
    -- Cantidades de transacciones (int)
    TRY_CAST(a.trx_q_abonos_1m_total AS INTEGER) AS trx_q_abonos_1m_total,
    TRY_CAST(a.trx_q_abonos_3m_total AS INTEGER) AS trx_q_abonos_3m_total,
    TRY_CAST(a.trx_q_abonos_6m_total AS INTEGER) AS trx_q_abonos_6m_total,
    TRY_CAST(a.trx_q_abonos_9m_total AS INTEGER) AS trx_q_abonos_9m_total,
    TRY_CAST(a.trx_q_abonos_12m_total AS INTEGER) AS trx_q_abonos_12m_total,
    TRY_CAST(a.trx_q_cargos_1m_total AS INTEGER) AS trx_q_cargos_1m_total,
    TRY_CAST(a.trx_q_cargos_3m_total AS INTEGER) AS trx_q_cargos_3m_total,
    TRY_CAST(a.trx_q_cargos_6m_total AS INTEGER) AS trx_q_cargos_6m_total,
    TRY_CAST(a.trx_q_cargos_9m_total AS INTEGER) AS trx_q_cargos_9m_total,
    TRY_CAST(a.trx_q_cargos_12m_total AS INTEGER) AS trx_q_cargos_12m_total,
    TRY_CAST(a.trx_q_abonos_1m_efectivo AS INTEGER) AS trx_q_abonos_1m_efectivo,
    TRY_CAST(a.trx_q_abonos_3m_efectivo AS INTEGER) AS trx_q_abonos_3m_efectivo,
    TRY_CAST(a.trx_q_abonos_6m_efectivo AS INTEGER) AS trx_q_abonos_6m_efectivo,
    TRY_CAST(a.trx_q_abonos_9m_efectivo AS INTEGER) AS trx_q_abonos_9m_efectivo,
    TRY_CAST(a.trx_q_abonos_12m_efectivo AS INTEGER) AS trx_q_abonos_12m_efectivo,
    TRY_CAST(a.trx_q_cargos_1m_efectivo AS INTEGER) AS trx_q_cargos_1m_efectivo,
    TRY_CAST(a.trx_q_cargos_3m_efectivo AS INTEGER) AS trx_q_cargos_3m_efectivo,
    TRY_CAST(a.trx_q_cargos_6m_efectivo AS INTEGER) AS trx_q_cargos_6m_efectivo,
    TRY_CAST(a.trx_q_cargos_9m_efectivo AS INTEGER) AS trx_q_cargos_9m_efectivo,
    TRY_CAST(a.trx_q_cargos_12m_efectivo AS INTEGER) AS trx_q_cargos_12m_efectivo,

    -- Promedios y ratios (ya estaban casi todos)
    TRY_CAST(a.trx_q_abonos_promedio_3m_total AS DOUBLE) AS trx_q_abonos_promedio_3m_total,
    TRY_CAST(a.trx_q_abonos_promedio_6m_total AS DOUBLE) AS trx_q_abonos_promedio_6m_total,
    TRY_CAST(a.trx_q_abonos_promedio_9m_total AS DOUBLE) AS trx_q_abonos_promedio_9m_total,
    TRY_CAST(a.trx_q_abonos_promedio_12m_total AS DOUBLE) AS trx_q_abonos_promedio_12m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_3m_total AS DOUBLE) AS trx_monto_abonos_promedio_3m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_6m_total AS DOUBLE) AS trx_monto_abonos_promedio_6m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_9m_total AS DOUBLE) AS trx_monto_abonos_promedio_9m_total,
    TRY_CAST(a.trx_monto_abonos_promedio_12m_total AS DOUBLE) AS trx_monto_abonos_promedio_12m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_3m_total AS DOUBLE) AS trx_monto_cargos_promedio_3m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_6m_total AS DOUBLE) AS trx_monto_cargos_promedio_6m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_9m_total AS DOUBLE) AS trx_monto_cargos_promedio_9m_total,
    TRY_CAST(a.trx_monto_cargos_promedio_12m_total AS DOUBLE) AS trx_monto_cargos_promedio_12m_total,
    TRY_CAST(a.trx_q_cargos_promedio_3m_total AS DOUBLE) AS trx_q_cargos_promedio_3m_total,
    TRY_CAST(a.trx_q_cargos_promedio_6m_total AS DOUBLE) AS trx_q_cargos_promedio_6m_total,
    TRY_CAST(a.trx_q_cargos_promedio_9m_total AS DOUBLE) AS trx_q_cargos_promedio_9m_total,
    TRY_CAST(a.trx_q_cargos_promedio_12m_total AS DOUBLE) AS trx_q_cargos_promedio_12m_total,

    -- Ratios efectivo/total
    TRY_CAST(a.trx_q_abonos_ratio_1m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_3m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_6m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_9m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_q_abonos_ratio_12m_efectivo_total AS DOUBLE) AS trx_q_abonos_ratio_12m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_1m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_3m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_6m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_9m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_monto_abonos_ratio_12m_efectivo_total AS DOUBLE) AS trx_monto_abonos_ratio_12m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_1m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_3m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_6m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_9m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_q_cargos_ratio_12m_efectivo_total AS DOUBLE) AS trx_q_cargos_ratio_12m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_1m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_1m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_3m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_3m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_6m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_6m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_9m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_9m_efectivo_total,
    TRY_CAST(a.trx_monto_cargos_ratio_12m_efectivo_total AS DOUBLE) AS trx_monto_cargos_ratio_12m_efectivo_total,

    -- Máximos, diferencias, RO, exterior, etc.
    TRY_CAST(a.dif_monto_abonos_cargos_efectivo_6m AS DOUBLE) AS dif_monto_abonos_cargos_efectivo_6m,
    a.dif_q_abonos_cargos_efectivo_6m,
    TRY_CAST(a.monto_ro_debajo_umbral AS DOUBLE) AS monto_ro_debajo_umbral,
    TRY_CAST(a.trx_monto_abonos_1m_max AS DOUBLE) AS trx_monto_abonos_1m_max,
    TRY_CAST(a.trx_monto_abonos_3m_max AS DOUBLE) AS trx_monto_abonos_3m_max,
    TRY_CAST(a.trx_monto_abonos_6m_max AS DOUBLE) AS trx_monto_abonos_6m_max,
    TRY_CAST(a.trx_monto_abonos_9m_max AS DOUBLE) AS trx_monto_abonos_9m_max,
    TRY_CAST(a.trx_monto_abonos_12m_max AS DOUBLE) AS trx_monto_abonos_12m_max,
    TRY_CAST(a.trx_monto_cargos_1m_max AS DOUBLE) AS trx_monto_cargos_1m_max,
    TRY_CAST(a.trx_monto_cargos_3m_max AS DOUBLE) AS trx_monto_cargos_3m_max,
    TRY_CAST(a.trx_monto_cargos_6m_max AS DOUBLE) AS trx_monto_cargos_6m_max,
    TRY_CAST(a.trx_monto_cargos_9m_max AS DOUBLE) AS trx_monto_cargos_9m_max,
    TRY_CAST(a.trx_monto_cargos_12m_max AS DOUBLE) AS trx_monto_cargos_12m_max,

    TRY_CAST(a.monto_al_exterior_3m AS DOUBLE) AS monto_al_exterior_3m,
    TRY_CAST(a.monto_al_exterior_6m AS DOUBLE) AS monto_al_exterior_6m,
    TRY_CAST(a.monto_al_exterior_9m AS DOUBLE) AS monto_al_exterior_9m,
    TRY_CAST(a.monto_al_exterior_12m AS DOUBLE) AS monto_al_exterior_12m,
    TRY_CAST(a.monto_del_exterior_3m AS DOUBLE) AS monto_del_exterior_3m,
    TRY_CAST(a.monto_del_exterior_6m AS DOUBLE) AS monto_del_exterior_6m,
    TRY_CAST(a.monto_del_exterior_9m AS DOUBLE) AS monto_del_exterior_9m,
    TRY_CAST(a.monto_del_exterior_12m AS DOUBLE) AS monto_del_exterior_12m,

    -- Variables que ya venían limpias
    a.edad,
    a.edad_constitucion,
    a.antiguedad,
    a.cantidad_lsb,
    a.nivel_riesgo_lsb_total,
    a.nivel_riesgo_lsb_ultima,
    a.q_meses_ingresos_0,
    a.q_meses_egresos_0,
    a.provincia,
    a.departamento,
    a.ubigeo_cd,
    a.sectorista_id,
    a.ciiu_v4,

    -- Flags booleanos → 0/1
    CASE WHEN a.flag_casos_hist THEN 1 ELSE 0 END AS flag_casos_hist,
    CASE WHEN a.flag_desv_activa THEN 1 ELSE 0 END AS flag_desv_activa,
    CASE WHEN a.flag_ros_hist THEN 1 ELSE 0 END AS flag_ros_hist,
    CASE WHEN a.flag_variacion_abono_monto_total_5m_1m THEN 1 ELSE 0 END AS flag_variacion_abono_monto_total_5m_1m,
    CASE WHEN a.flag_variacion_efect_cargos_monto_5m_1m THEN 1 ELSE 0 END AS flag_variacion_efect_cargos_monto_5m_1m,

    -- Flags que ya son int
    a.flag_cce_r1,
    a.flag_kyc_12m,
    a.flag_kyc_hist,
    a.cantidad_kyc_hist,
    a.cp_cantidad_ing,
    a.q_ro_debajo_umbral,
    a.q_trx_al_exterior_3m, a.q_trx_al_exterior_6m, a.q_trx_al_exterior_9m, a.q_trx_al_exterior_12m,
    a.q_trx_del_exterior_3m, a.q_trx_del_exterior_6m, a.q_trx_del_exterior_9m, a.q_trx_del_exterior_12m,
    a.q_meses_al_exterior_0, a.q_meses_al_exterior_100, a.q_meses_al_exterior_1000,
    a.q_meses_al_exterior_10000, a.q_meses_al_exterior_100000, a.q_meses_al_exterior_1000000,
    a.flag_al_exterior,
    a.flag_del_exterior,
    a.flag_inteligo,

    -- NUEVAS COLUMNAS que faltaban
    a.nacionalidad,
    a.tipo_cliente_sensible,
    TRY_CAST(a.score_con_excepcion AS DOUBLE) AS score_con_excepcion,
    a.tipobancadesc,
    a.flag_casos_12m,
    a.estado_ultimo_caso,
    a.q_casos_hist,
    a.flag_desv_hist,
    a.estado_ultima_desv,
    a.periodo_inicio_desv_max,
    a.periodo_cierre_desv_max,
    a.meses_en_proceso_desvinculacion,
    a.flag_alerta_12m,
    a.flag_alerta_hist,
    a.calificacion_ultima_alerta,
    a.q_alerta_hist,
    a.flag_ros_12m,
    a.q_ros_hist,
    TRY_CAST(a.monto_trx_debajo_10k_ing AS DOUBLE) AS monto_trx_debajo_10k_ing,
    a.cant_trx_debajo_10k_ing,
    a.cant_trx_debajo_10k_egr,
    TRY_CAST(a.monto_trx_debajo_10k_egr AS DOUBLE) AS monto_trx_debajo_10k_egr,
    TRY_CAST(a.cp_cant_trx_egr_desv AS DOUBLE) AS cp_cant_trx_egr_desv,
    TRY_CAST(a.cp_monto_total_egr_desv AS DOUBLE) AS cp_monto_total_egr_desv,
    TRY_CAST(a.cp_cant_trx_ing_ros AS DOUBLE) AS cp_cant_trx_ing_ros,
    TRY_CAST(a.cp_monto_total_ing_ros AS DOUBLE) AS cp_monto_total_ing_ros,
    TRY_CAST(a.cp_cant_trx_egr_ros AS DOUBLE) AS cp_cant_trx_egr_ros,
    TRY_CAST(a.cp_monto_total_egr_ros AS DOUBLE) AS cp_monto_total_egr_ros,

    a.p_codmes,
    b.tipo_alerta_n2,

    -- TARGET
    CASE WHEN a.flg_alerta = '1' THEN 1 ELSE 0 END AS target_m


FROM d_perm_aws.ds_alertplaft_mdl a
LEFT JOIN e_perm_aws.t_alertas_plaft b
  ON a.codunico = b.codunico
 AND a.mes_base = b.periodo_alerta
WHERE a.mes_base BETWEEN '202501' AND '202509'
  AND a.subsegmento = 'BPE'

;"""
df = athena_query(query, database='disc_comercial')
df.head()

In [6]:
df_dataset.shape

(1426243, 187)

In [7]:
df= df_dataset

In [8]:
df.nivel_riesgo_lsb_total.value_counts()

nivel_riesgo_lsb_total
ALTO    791911
Name: count, dtype: Int64

In [9]:
df['nivel_riesgo_lsb_total'] = df['nivel_riesgo_lsb_total'].map({
    'ALTO':1,
    'MEDIO':2,
    'BAJO':3,
    }).fillna(-1)

In [10]:
df['nivel_riesgo_lsb_ultima'] = df['nivel_riesgo_lsb_ultima'].map({
    'ALTO':1,
    'MEDIO':2,
    'BAJO':3,
    }).fillna(-1)

In [11]:
df.flag_casos_hist.value_counts()

flag_casos_hist
0    1425477
1        766
Name: count, dtype: Int64

In [12]:
df["cantidad_lsb"] =df["cantidad_lsb"].fillna(0)

In [13]:
# df_dataset: datos correspondientes a 202405
# df_dataset = pd.read_csv('C:/Users/X15403/Desktop/PLAFT/exploratorio/t_dataset_plaft_202405.csv', encoding='utf-8', sep=',', low_memory=False)

#df_dataset = pd.read_csv('C:/Users/X15403/Desktop/PLAFT/exploratorio/preprocesamiento.csv', encoding='utf-8', sep=',', low_memory=False)
#df_dataset = df_dataset.rename(columns={'flg_alerta': 'target_m'})
#df_dataset = df_dataset.drop_duplicates()

In [14]:
df.tipo_alerta_n2.value_counts()

tipo_alerta_n2
AUTOMATICA         4514
MANUAL              765
SEMI AUTOMATICA     537
Name: count, dtype: Int64

# Definición de funciones

In [15]:
# Funcion columnas nulas
########################

def filtro_columnas_nulas(df: pd.DataFrame,
                          umbral_nulos: float,
                          col_conservar: List[str] = None,
                          col_eliminar: List[str] = None,
                          col_target: str = 'target_m'
                         ) -> pd.DataFrame:
    """
    Elimina columnas con más del porcentaje de nulos especificado, columnas con un único valor, y filas duplicadas.
    Las columnas en 'col_conservar' y la columna objetivo 'col_target' nunca se eliminan si existen en el DataFrame.

    Parámetros:
    - df: DataFrame a limpiar.
    - umbral_nulos: Porcentaje máximo permitido de nulos por columna (ej. 0.2 para 20%).
    - col_conservar: Lista de nombres de columnas que deben conservarse si existen.
    - col_conservar: Lista de nombres de columnas a eliminar.
    - col_target: Nombre de la columna objetivo que debe conservarse.

    Retorna:
    - DataFrame limpio.
    """
    print("\n\033[1m\033[4mFILTRANDO COLUMNAS...\033[0m\n")
    print(f"Shape inicial del df: {df.shape}\n")

    # Paso 0: Eliminar columnas especificadas en col_eliminar si existen
    if col_eliminar:
        cols_eliminar_existentes = [col for col in col_eliminar if col in df.columns]
        if cols_eliminar_existentes:
            df = df.drop(columns=cols_eliminar_existentes)
            print("Se eliminan del df las siguientes columnas:")
            for col in cols_eliminar_existentes:
                print(f" - {col}")
            print("")
        else:
            print("Ninguna de las columnas especificadas en col_eliminar existe en el DataFrame.\n")
    
    # Validación del umbral
    if not 0 <= umbral_nulos <= 1:
        raise ValueError("El umbral de nulos debe estar entre 0 y 1.")

    # Inicializar lista de columnas a conservar
    col_conservar = col_conservar or []

    # Asegurar que col_target esté en la lista de columnas a conservar
    if col_target not in col_conservar:
        col_conservar.append(col_target)

    # Filtrar solo las columnas que existen en el DataFrame
    col_conservar_existentes = [col for col in col_conservar if col in df.columns]

    # Mensaje de columnas que se conservarán
    print("Listado de columnas a conservar (no se aplica filtro segun porcentaje de datos faltates):")
    for col in col_conservar_existentes:
        print(f" - {col}")

    # Eliminar columnas con un único valor, preservando las columnas a conservar
    columnas_valor_unico = [
        col for col in df.columns
        if df[col].nunique() == 1 and col not in col_conservar_existentes
    ]
    print("")
    print("Se eliminan las siguientes columnas por tener valor único:")
    for col in columnas_valor_unico:
        print(f" - {col}")
        
    df = df.drop(columns=columnas_valor_unico)

    # Calcular porcentaje de nulos por columna
    porcentaje_nulos = df.isnull().mean()

    # Filtrar columnas que cumplen el umbral, preservando las columnas a conservar
    columnas_validas = [
        col for col in df.columns
        if (porcentaje_nulos[col] < umbral_nulos or col in col_conservar_existentes)
    ]
    df_filtrado = df[columnas_validas]
    print(f"\nSe eliminan {str(len(df.columns) - len(df_filtrado.columns))} columnas con más del {umbral_nulos*100}% de nulos")

    # Eliminar filas duplicadas
    df_filtrado = df_filtrado.drop_duplicates()
    
    print(f"\nShape del df: {df_filtrado.shape}")

    return df_filtrado

In [16]:
# Funcion target
################

def generar_target(df: pd.DataFrame, escenario: int, porcentaje_sampleo: float = 0.1) -> pd.DataFrame:
    """
    Genera o modifica la columna 'target_m' en el DataFrame según el escenario elegido.

    Escenarios:
    1 - Elimina registros con target nulo y convierte a entero.
    2 - Mantiene alertas con riesgo y agrega muestra de casos sin alertas (target nulo).
    3 - Solo mantiene alertas con riesgo y muestra de casos sin alertas.

    Parámetros:
    - df: DataFrame original.
    - escenario: Número de escenario (1, 2 o 3).
    - porcentaje_sampleo: Porcentaje de registros nulos a muestrear (valor entre 0 y 1).

    Retorna:
    - DataFrame modificado con columna 'target_m' procesada.
    """
    print("\n\033[1m\033[4mCREANDO TARGET...\033[0m\n")

    print("Escenarios posibles de creacion de target: ")
    print("1 - Elimina registros con target nulo y convierte a entero.")
    print("2 - Mantiene alertas con riesgo y agrega muestra de casos sin alertas (target nulo).")
    print("3 - Solo mantiene alertas con riesgo y muestra de casos sin alertas.")

    print(f"\nEscenario elegido : {escenario}")
    
    if escenario not in [1, 2, 3]:
        raise ValueError("El escenario debe ser 1, 2 o 3.")
    if not 0 < porcentaje_sampleo <= 1:
        raise ValueError("El porcentaje de sampleo debe estar entre 0 y 1.")

    df = df.copy()

    if escenario == 1:
        # Eliminar nulos y convertir a entero
        df = df[~df['target_m'].isnull()]
        df['target_m'] = df['target_m'].astype(int)

    else:
        # Escenarios 2 y 3: muestreo de registros con target nulo
        columns_to_check = [col for col in df.columns if col != 'target_m']
        sample_df = df[df['target_m'].isnull()].copy()
        sample_df['null_count'] = sample_df[columns_to_check].isnull().sum(axis=1)
        sorted_sample = sample_df.sort_values(by='null_count')

        n = int(len(sample_df) * porcentaje_sampleo)
        df_null = sorted_sample.head(n)

        if escenario == 2:
            df = pd.concat([df[~df['target_m'].isnull()], df_null])
        elif escenario == 3:
            df_target_1 = df[df['target_m'] == 1]
            df = pd.concat([df_target_1, df_null])

        df['target_m'] = df['target_m'].fillna(0)
        df['target_m'] = df['target_m'].astype(int)

    df = df.drop(columns=['null_count'])

    print(f"\nShape del df: {df.shape}")
    print("\nDistribución de la target:")
    print(df['target_m'].value_counts())

    return df



In [17]:
# Funcion para clasificar columnas
##################################

def clasificar_columnas(df: pd.DataFrame, 
                        target_variable: str):
    """
    Clasifica las columnas de un DataFrame en listas de variables target, fechas, categóricas y numéricas.
    Elimina previamente columnas que tengan valor unico.
    Elimina filas duplicadas.

    Parámetros:
    - df (pd.DataFrame): DataFrame de entrada.
    - target_variable (str): Nombre de la variable objetivo.
    - col_eliminar (list, opcional): Lista de columnas a eliminar explícitamente.

    Retorna:
    tuple: col_target, col_fechas, col_categoricas, col_numericas, df (modificado)
    """
    print("\n\033[1m\033[4mCLASIFICANDO COLUMNAS...\033[0m\n")

    # Paso 1: Eliminar columnas que tienen un único valor
    cols_to_drop = [col for col in df.columns if df[col].nunique() == 1]
    print("Se eliminan las siguientes columnas por tener valor único:")
    for col in cols_to_drop:
        print(f" - {col}")

    if 'key_value' in df.columns:
        cols_to_drop.append('key_value')
    df = df.drop(columns=cols_to_drop)

    print("")
    print(f"Shape después de eliminar columnas: {df.shape}")

    # Paso 2: Eliminar filas duplicadas
    df = df.drop_duplicates()
    print(f"Shape después de eliminar filas duplicadas: {df.shape}")
    print("")

    # Paso 3: Identificar la variable target
    col_target = [target_variable] if target_variable in df.columns else []

    # Paso 4: Identificar variables categóricas y numéricas
    col_categoricas = df.select_dtypes(include='object').columns.tolist()
    col_categoricas = [col for col in col_categoricas if col not in col_target]

    col_numericas = df.select_dtypes(include=['int64', 'float64','int32','number']).columns.tolist()
    col_numericas = [col for col in col_numericas if col not in col_target]

    # Paso 5: Corrección de clasificación usando substrings
    substrings_categoricas = ['flag', 'lugar', 'tipo', 'cod', 'ciiu_v4', 'ubigeo_cd']
    moved_to_categoricas = [col for col in col_numericas if any(sub in col.lower() for sub in substrings_categoricas)]
    col_categoricas.extend(moved_to_categoricas)
    col_numericas = [col for col in col_numericas if col not in moved_to_categoricas]

    # Paso 5.2: Corrección de clasificación usando substrings
    substrings_numericas = ['v13_lugar_operativa_riesgo', 'v16_canal_operacion_riesgo']
    moved_to_numericas = [col for col in col_categoricas if any(sub in col.lower() for sub in substrings_numericas)]
    col_numericas.extend(moved_to_numericas)
    col_categoricas = [col for col in col_categoricas if col not in moved_to_numericas]

    # Paso 6: Identificación de variables de fecha
    col_fechas = []
    substrings_fecha = ['fecha', 'mes_base']
    moved_from_categoricas = [col for col in col_categoricas if any(sub in col.lower() for sub in substrings_fecha)]
    col_fechas.extend(moved_from_categoricas)
    col_categoricas = [col for col in col_categoricas if col not in col_fechas]

    moved_from_numericas = [col for col in col_numericas if any(sub in col.lower() for sub in substrings_fecha)]
    col_fechas.extend(moved_from_numericas)
    col_numericas = [col for col in col_numericas if col not in col_fechas]

    # Paso 7: Reemplazar '0.0' por '0' y '1.0' por '1' en columnas categóricas de flag 
    for col in col_categoricas:
        df[col] = df[col].astype('str')
        if col in df.columns and 'flag_' in col:
            df[col] = df[col].replace({'0.0': '0', '1.0': '1'})

    # Impresión de listas clasificadas con cantidad de elementos
    print(f"Variables de fecha ({len(col_fechas)}):")
    for col in col_fechas:
        print(f" - {col}")
    print()

    print(f"Variables categóricas ({len(col_categoricas)}):")
    for col in col_categoricas:
        print(f" - {col}")
    print()

    print(f"Variables numéricas ({len(col_numericas)}):")
    for col in col_numericas:
        print(f" - {col}")
    print()

    # Retornar las listas y el DataFrame modificado
    return col_target, col_fechas, col_categoricas, col_numericas, df

In [18]:
# Funcion para imputar nulos en columnas categoricas
####################################################

def imputar_categoricas(df: pd.DataFrame,
                        col_categoricas: List[str],
                        criterio_riesgo: str = 'sin_dato',
                        criterio_flag: str = 'sin_dato',
                        criterio_cat: str = 'sin_dato'
                       ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Imputa valores faltantes en columnas categóricas según criterios definidos.

    Parámetros:
    - df: DataFrame original.
    - col_categoricas: Lista de nombres de columnas categóricas.
    - criterio_riesgo: Valor para imputar columnas que contienen 'nivel_riesgo'.
    - criterio_flag: Valor para imputar columnas que contienen 'flag_'.
    - criterio_cat: Valor para imputar el resto ('sin_dato' o 'moda').

    Retorna:
    - df_imputado: DataFrame con imputaciones realizadas.
    - df_resumen: DataFrame resumen con criterios y cantidad de imputaciones por columna.
    """
    print("\n\033[1m\033[4mIMPUTANDO NULOS PARA COLUMNAS CATEGORICAS...\033[0m\n")
    df_imputado = df.copy()
    resumen = []

    for col in col_categoricas:
        if col not in df_imputado.columns:
            continue  # Saltear columnas que no existen en el DataFrame

        # Convertir a string para asegurar tipo
        df_imputado[col] = df_imputado[col].astype(str)

        nulos_antes = df_imputado[col].isna().sum() + (df_imputado[col] == 'nan').sum()

        if 'nivel_riesgo' in col:
            valor_imputacion = criterio_riesgo
            df_imputado[col] = df_imputado[col].replace('nan', pd.NA)
            df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        elif 'flag_' in col:
            valor_imputacion = criterio_flag
            df_imputado[col] = df_imputado[col].replace('nan', pd.NA)
            df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        else:
            if criterio_cat == 'sin_dato':
                valor_imputacion = 'sin_dato'
            elif criterio_cat == 'moda':
                moda = df_imputado[col].mode(dropna=True)
                valor_imputacion = moda[0] if not moda.empty else 'sin_dato'
            else:
                valor_imputacion = 'sin_dato'  # fallback

            df_imputado[col] = df_imputado[col].replace('nan', pd.NA)
            df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        # Asegurar tipo string después de imputar
        df_imputado[col] = df_imputado[col].astype(str)

        nulos_despues = df_imputado[col].isna().sum() + (df_imputado[col] == 'nan').sum()
        imputados = nulos_antes - nulos_despues

        resumen.append({
            'columna': col,
            'criterio_usado': valor_imputacion,
            'valores_imputados': imputados
        })

    df_resumen = pd.DataFrame(resumen)

    print('Resumen\n')
    display(df_resumen)

    return df_imputado, df_resumen


In [19]:
# Funcion para imputar nulos en columnas numércias
##################################################

def imputar_numericas(df: pd.DataFrame,
                      col_numericas: List[str],
                      criterio_dif: str,
                      criterio_trx: str,
                      criterio_q: str,
                      criterio_num: str
                     ) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Imputa valores nulos en columnas numéricas según criterios definidos por tipo de variable.

    Parámetros:
    ----------
    df : pd.DataFrame
        DataFrame original.
    col_numericas : List[str]
        Lista de nombres de columnas numéricas a imputar.
    criterio_dif : str
        Criterio para columnas que contienen 'dif_'. Opciones: '0', 'media', 'max', 'p50'.
    criterio_trx : str
        Criterio para columnas que contienen 'trx_'. Opciones: '0', 'media', 'max', 'p50'.
    criterio_q : str
        Criterio para columnas que contienen 'q_'. Opciones: '0', 'media', 'max', 'p50'.
    criterio_num : str
        Criterio para el resto de las columnas. Opciones: '0', 'media', 'max', 'p50'.

    Retorna:
    -------
    df_imputado : pd.DataFrame
        DataFrame con valores imputados.
    df_resumen : pd.DataFrame
        DataFrame resumen con variable, criterio usado y cantidad de valores imputados.
    """
    print("\n\033[1m\033[4mIMPUTANDO NULOS PARA COLUMNAS NUMERICAS...\033[0m\n")
    df_imputado = df.copy()
    resumen = []

    criterios_validos = {"0", "media", "max", "p50"}

    def obtener_valor_imputacion(col: pd.Series, criterio: str):
        if criterio not in criterios_validos:
            raise ValueError(f"Criterio inválido: {criterio}")
        if criterio == "0":
            return 0
        elif criterio == "media":
            return col.mean()
        elif criterio == "max":
            return col.max()
        elif criterio == "p50":
            return col.quantile(0.5)

    for col in col_numericas:
        if col not in df_imputado.columns:
            continue  # O podrías loggear una advertencia
        if not pd.api.types.is_numeric_dtype(df_imputado[col]):
            continue

        nulos_antes = df_imputado[col].isna().sum()
        if nulos_antes == 0:
            continue

        if "dif_" in col:
            criterio = criterio_dif
        elif "trx_" in col:
            criterio = criterio_trx
        elif "q_" in col:
            criterio = criterio_q
        else:
            criterio = criterio_num

        valor_imputacion = obtener_valor_imputacion(df_imputado[col], criterio)
        df_imputado[col] = df_imputado[col].fillna(valor_imputacion)

        resumen.append({
            "variable": col,
            "criterio_usado": criterio,
            "valores_imputados": nulos_antes
        })

    df_resumen = pd.DataFrame(resumen)
    print('Resumen\n')
    display(df_resumen)
    return df_imputado, df_resumen

In [20]:
# Funcion para reagrupar variables categoricas
##############################################

def reagrupar_categorias(df, col_categoricas, target_m):
    """
    Reagrupa niveles de variables categóricas segun cantidad de casos positivos y efectividad.

    Parámetros:
    ----------
    df : pd.DataFrame
        DataFrame original.
    col_categoricas : List[str]
        Lista de nombres de columnas categóricas a reagrupar.
    target_m : str
        Variable target.

    Retorna:
    -------
    df_resultado : pd.DataFrame
        DataFrame con valores reagrupados.
    df_resumen : pd.DataFrame
        DataFrame resumen con variable, nuevo grupo, categoría original, detalle.
    diccionario:
        variabe: nuevo grupo: categoria original
    """
    print("\n\033[1m\033[4mREAGRUPANDO VARIABLES CATEGORICAS...\033[0m\n")
    
    df_resultado = df.copy()
    resumen = []

    # eliminamos tipo de alerta de las columnas categoricas
    if 'tipo_alerta_n2' in col_categoricas:
        col_categoricas.remove('tipo_alerta_n2')

    for col in col_categoricas:
        categorias = df[col].dropna().unique()
        if len(categorias) <= 3:
            continue
        # Calcular casos positivos y negativos por categoría
        stats = df.groupby(col)[target_m].agg(['sum', 'count'])
        stats['casos_positivos'] = stats['sum']
        stats['casos_negativos'] = stats['count'] - stats['sum']
        stats['efectividad'] = stats.apply(
            lambda row: row['casos_positivos'] / row['casos_negativos'] if row['casos_negativos'] > 0 else float('inf'),
            axis=1
        )
        stats = stats[['casos_positivos', 'casos_negativos', 'efectividad']]
        stats = stats.reset_index()

        # Escenario 1: todas las categorías tienen casos_positivos = 0
        if (stats['casos_positivos'] == 0).all():
            mapping = {cat: f"{col}_nulo" for cat in categorias}
            resumen.append({
                'variable': col,
                'grupo': f"{col}_nulo",
                'categorias_originales': list(categorias),
                'detalle': 'Todas las categorías con casos_positivos = 0'
            })

        # Escenario 2: algunas categorías con casos_positivos = 0, otras > 0
        elif (stats['casos_positivos'] == 0).any():
            nulo_cats = stats[stats['casos_positivos'] == 0][col].tolist()
            positivas = stats[stats['casos_positivos'] > 0].sort_values(by='efectividad', ascending=False)
            pos_cats = positivas[col].tolist()

            mapping = {cat: f"{col}_nulo" for cat in nulo_cats}

            if len(pos_cats) == 1:
                mapping[pos_cats[0]] = f"{col}_medio"
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': [pos_cats[0]],
                    'detalle': 'Una categoría con casos_positivos > 0'
                })
            else:
                mitad = len(pos_cats) // 2
                alto = pos_cats[:mitad]
                medio = pos_cats[mitad:]
                for cat in alto:
                    mapping[cat] = f"{col}_alto"
                for cat in medio:
                    mapping[cat] = f"{col}_medio"
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_alto",
                    'categorias_originales': alto,
                    'detalle': 'Categorías con alta efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': medio,
                    'detalle': 'Categorías con efectividad media'
                })
            resumen.append({
                'variable': col,
                'grupo': f"{col}_nulo",
                'categorias_originales': nulo_cats,
                'detalle': 'Categorías con casos_positivos = 0'
            })

        # Escenario 3: todas las categorías tienen casos_positivos >= 1
        else:
            stats_sorted = stats.sort_values(by='efectividad', ascending=False)
            sorted_cats = stats_sorted[col].tolist()
            n = len(sorted_cats)

            if n == 1:
                mapping = {sorted_cats[0]: f"{col}_medio"}
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': [sorted_cats[0]],
                    'detalle': 'Una sola categoría con casos_positivos >= 1'
                })
            elif n == 2:
                mapping = {
                    sorted_cats[0]: f"{col}_alto",
                    sorted_cats[1]: f"{col}_bajo"
                }
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_alto",
                    'categorias_originales': [sorted_cats[0]],
                    'detalle': 'Categoría con mayor efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_bajo",
                    'categorias_originales': [sorted_cats[1]],
                    'detalle': 'Categoría con menor efectividad'
                })
            else:
                tercio = n // 3
                alto = sorted_cats[:tercio]
                medio = sorted_cats[tercio:2*tercio]
                bajo = sorted_cats[2*tercio:]
                mapping = {}
                for cat in alto:
                    mapping[cat] = f"{col}_alto"
                for cat in medio:
                    mapping[cat] = f"{col}_medio"
                for cat in bajo:
                    mapping[cat] = f"{col}_bajo"
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_alto",
                    'categorias_originales': alto,
                    'detalle': 'Tercio superior de efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_medio",
                    'categorias_originales': medio,
                    'detalle': 'Tercio medio de efectividad'
                })
                resumen.append({
                    'variable': col,
                    'grupo': f"{col}_bajo",
                    'categorias_originales': bajo,
                    'detalle': 'Tercio inferior de efectividad'
                })

        # Aplicar el mapeo al DataFrame
        df_resultado[col] = df[col].map(mapping).fillna(df[col])

    df_resumen = pd.DataFrame(resumen)

    print('Resumen\n')
    display(df_resumen)

    # diccionario con las categorias reagrupadas
    diccionario = {}
    
    for _, row in df_resumen.iterrows():
        variable = row['variable']
        grupo = row['grupo']
        categorias = row['categorias_originales']
        
        if variable not in diccionario:
            diccionario[variable] = {}
        
        diccionario[variable][grupo] = categorias

    return df_resultado, df_resumen, diccionario

In [21]:
# Funcion para eliminar variables altamente correlacionadas
###########################################################

# limpieza por correlación
def _prepare_numeric(df_sub: pd.DataFrame) -> pd.DataFrame:
   return df_sub.apply(pd.to_numeric, errors="coerce")


def _is_constant(series: pd.Series) -> bool:
   # constante si menos de 2 valores no-NaN distintos
   return series.dropna().nunique() < 2


def _safe_corr_with_target(df_num: pd.DataFrame, 
                           feats: list, 
                           target: str, 
                           method: str) -> dict:
   out = {}
   tgt = df_num[target]
   tgt_const = _is_constant(tgt)
   for f in feats:
       s = df_num[f]
       if _is_constant(s) or tgt_const:
           out[f] = 0.0
       else:
           with np.errstate(invalid='ignore', divide='ignore'):
               val = s.corr(tgt, method=method)
           out[f] = 0.0 if (pd.isna(val) or np.isinf(val)) else abs(val)
   return out


def _pair_corr(df_num: pd.DataFrame, 
               a: str, 
               b: str, 
               method: str) -> float:
   sa, sb = df_num[a], df_num[b]
   # si cualquiera es constante, define corr=0 (no dispara eliminación)
   if _is_constant(sa) or _is_constant(sb):
       return 0.0
   pair = pd.concat([sa, sb], axis=1).dropna()
   if pair.shape[0] < 2:
       return 0.0
   with np.errstate(invalid='ignore', divide='ignore'):
       c = pair.corr(method=method).abs().iloc[0, 1]
   if pd.isna(c) or np.isinf(c):
       return 0.0
   return float(c)


def _drops_by_method(df_num: pd.DataFrame, 
                     feats: list, 
                     target: str, 
                     limit: float, 
                     method: str) -> list:
   if len(feats) <= 1 or limit is None:
       return []
   to_remove = set()
   # |corr(feature, target)|
   corr_with_target = _safe_corr_with_target(df_num, feats, target, method)
   n = len(feats)
   corr_dict = {}
   for i in range(n):
       fi = feats[i]
       corr_feature = {}
       if fi in to_remove:
           continue
       for j in range(i + 1, n):
           fj = feats[j]
           if fj in to_remove:
               continue
           corr = _pair_corr(df_num, fi, fj, method)
           corr_feature[fj] = corr 
           if corr >= limit:
               # desempate: conservar mayor |corr con target|
               if corr_with_target.get(fi, 0.0) >= corr_with_target.get(fj, 0.0):
                   to_remove.add(fj)
                   # print(f"Removed {fj} because: {fj}: {corr_with_target.get(fi, 0.0)} >= {corr_with_target.get(fj, 0.0)}")
               else:
                   to_remove.add(fi)
                   # print(f"Removed {fi} because: {fj}: {corr_with_target.get(fi, 0.0)} < {corr_with_target.get(fj, 0.0)}")
                   break
       corr_dict[fi] = corr_feature
   return list(to_remove), corr_dict
    
def select_features_by_correlation(
   df: pd.DataFrame,
   features: list,
   target_column: str,
   pearson_limit: float = 0.9,
   spearman_limit: float = 0.9,
   verbose: bool = True
) -> list:
   """
   Elimina features altamente correlacionadas en dos etapas:
     1) Pearson con umbral `pearson_limit`
     2) Spearman con umbral `spearman_limit`
   Desempate: conserva la que tenga mayor |corr(feature, target)| con el mismo método.
   Manejo extra:
     - Convierte a numérico (errores -> NaN).
     - Ignora columnas constantes o completamente NaN al calcular correlaciones (trata su corr como 0).
   Retorna la lista de features conservadas.
   """
   print("\n\033[1m\033[4mELIMINANDO COLUMNAS NUMERICAS ALTAMENTE CORRELACIONADAS...\033[0m\n")
   cols_needed = list(dict.fromkeys(list(features) + [target_column]))
   df_num = _prepare_numeric(df[cols_needed].copy())

   if verbose:
       const_cols = [c for c in cols_needed if _is_constant(df_num[c])]
       if const_cols:
           print(f"Columnas constantes/degeneradas (tratadas con corr=0): {const_cols}")
   
   # Etapa 1: Pearson
   remaining = sorted(set(features))
   drops_pearson, dict1 = _drops_by_method(df_num, remaining, target_column, pearson_limit, "pearson")
   
   remaining = sorted(set(remaining) - set(drops_pearson))
   # print('Pearson_remaining', remaining)
   print("Columnas eliminadas - correlación Pearson")
   for col in drops_pearson:
       print(f" - {col}")
    
   # Etapa 2: Spearman
   drops_spearman, dict2 = _drops_by_method(df_num, remaining, target_column, spearman_limit, "spearman")
   remaining = sorted(set(remaining) - set(drops_spearman))
   #print('Spearman_remaining', remaining)
   print("\nColumnas eliminadas - correlación Spearman")
   for col in drops_spearman:
       print(f" - {col}")
       
   drops = sorted(set(drops_pearson + drops_spearman))
   print('\nTotal columnas eliminadas: ' + str(len(drops)))

   df = df.drop(columns=drops)
   print(f"\nShape final del DataFrame: {df.shape}")
   print("\nDistribucion de la target:")
   print(df.target_m.value_counts())
   return df

In [22]:
# Funcion para crear nuevas columnas a partir de las columnas de fechas
#######################################################################

def procesar_fechas(df: pd.DataFrame, col_fechas: list) -> tuple:
    """
    Procesa columnas de fechas en un DataFrame:
    - Convierte columnas en col_fechas a datetime.
    - Calcula diferencias en días entre fechas específicas.
    - Devuelve:
        - Un DataFrame modificado con nuevas columnas y sin col_fechas originales.
        - Un listado con las nuevas columnas + fecha_vinculacion + fecha_constitucion.
    """
    print("\n\033[1m\033[4mCREANDO NUEVAS COLUMNAS FECHAS...\033[0m\n")
    
    df = df.copy()  # Evitar modificar el original

    # 1. Convertir columnas en col_fechas a datetime si existen
    for col in col_fechas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce')

    nuevas_columnas = []

    # 2. Diferencia entre fecha_constitucion y fecha_vinculacion
    if 'fecha_vinculacion' in df.columns and 'fecha_constitucion' in df.columns:
        df['dias_entre_constitucion_y_vinculacion'] = (
            df['fecha_vinculacion'] - df['fecha_constitucion']
        ).dt.days
        nuevas_columnas.append('dias_entre_constitucion_y_vinculacion')

    # 3. Diferencia entre fecha mínima de columnas cp_ y fecha_vinculacion
    cp_cols = [col for col in col_fechas if 'cp_' in col and col in df.columns]
    if cp_cols and 'fecha_vinculacion' in df.columns:
        df['cp_fecha_min'] = df[cp_cols].min(axis=1)
        df['dias_entre_cp_min_y_vinculacion'] = (
            df['cp_fecha_min'] - df['fecha_vinculacion']
        ).dt.days
        nuevas_columnas.append('dias_entre_cp_min_y_vinculacion')
        df.drop(columns=['cp_fecha_min'], inplace=True)

    # 4. Diferencia entre fecha mínima de otras columnas y fecha_vinculacion
    otras_cols = [
        col for col in col_fechas
        if col not in cp_cols + ['fecha_constitucion', 'fecha_vinculacion']
        and col in df.columns
    ]
    if otras_cols and 'fecha_vinculacion' in df.columns:
        df['otras_fecha_min'] = df[otras_cols].min(axis=1)
        df['dias_entre_min_fecha_y_vinculacion'] = (
            df['otras_fecha_min'] - df['fecha_vinculacion']
        ).dt.days
        nuevas_columnas.append('dias_entre_min_fecha_y_vinculacion')
        df.drop(columns=['otras_fecha_min'], inplace=True)

    # 5. Eliminar col_fechas del DataFrame
    cols_a_eliminar = [col for col in col_fechas if col in df.columns and col not in ['fecha_constitucion', 'fecha_vinculacion']]
    df.drop(columns=cols_a_eliminar, inplace=True)

    # Calcular estadísticas descriptivas
    df_resumen = df[nuevas_columnas].describe().T

    print('Resumen\n')
    display(df_resumen)

    # agregamos columnas a la lista de columnas fecha
    nuevas_columnas.extend(['fecha_constitucion', 'fecha_vinculacion'])
    
    # 7. Retornar DataFrame modificado y listado de columnas
    return df, nuevas_columnas, df_resumen

# Pre-procesamiento de datos

In [54]:
print("\033[1mPRE-PROCESANDO DF\033[0m\n")

# df
df_1 = df.copy()
# columnas que por decision del negocio deben conservarse para el entrenamiento del modelo
col_conservar = ['num_documento','cp_monto_total_ing', 'cp_promedio_mensual_ing', 'cp_maximo_mensual_ing', 'cp_ultimo_mes_ing', 'cp_monto_total_egr',
                 'cp_promedio_mensual_egr', 'cp_maximo_mensual_egr', 'cp_ultimo_mes_egr', 'cp_ratio_egr', 'cp_monto_pep_ing', 'cp_monto_pep_egr',
                 'cantidad_noticias', 'cp_flag_noticia_ing', 'cp_flag_noticia_egr', 'nivel_riesgo_lsb_total', 'nivel_riesgo_lsb_ultima',
                 'fecha_mas_antiguo_lsb', 'fecha_mas_reciente_lsb', 'entidad_solc_ultimo_lsb', 'cp_fecha_mas_antiguo_lsb_ing',
                 'cp_fecha_mas_reciente_lsb_ing', 'cp_fecha_mas_antiguo_lsb_egr', 'cp_fecha_mas_reciente_lsb_egr', 'cp_cantidad_lsb_egr',
                 'flag_ros_12m', 'flag_ros_hist', 'q_ros_hist', 'cantidad_lsb', 'cp_fecha_ultimo_ros_ing', 'cp_flag_ros_egr','cp_fecha_ultimo_ros_egr',
                 'cp_cantidad_lsb_egr', 'monto_ro_debajo_umbral', 'q_ro_debajo_umbral', 'cp_cant_trx_ing_ros', 'cp_monto_total_ing_ros',
                 'cp_cant_trx_egr_ros', 'cp_monto_total_egr_ros', 'flag_alerta_hist', 'q_alerta_hist', 'cp_flag_desv_egr', 'tipo_alerta_n2','flag_casos_hist'
                 'cantidad_lsb',
 'nivel_riesgo_lsb_total',
 'nivel_riesgo_lsb_ultima','monto_cargos_efectivo_ult_mes'
                ]

# columnas a eliminar
col_eliminar = ['codunico', 'coddocrele', 'key_value', 'target',  'p_codmes']

# columnas de control
col_control = ['target_m', 'mes_base', 'codunico']

# PRE-PROCESADO
df_1 = filtro_columnas_nulas(df_1, 0.2, col_conservar, col_eliminar, 'target_m') # filtramos columnas con alto porcentaje de nulos (conservando col_conservar)
df_1 = generar_target(df_1, 2, 0.1) # generamos la target
col_target, col_fechas, col_categoricas, col_numericas, df_1 = clasificar_columnas(df_1, 'target_m') # clasificamos columnas
df_1, resumen_imputacion_categoricas = imputar_categoricas(df_1,col_categoricas, 'BAJO', '0','sin_dato') # imputamos variables categoricas
df_1, resumen_imputacion_numericas = imputar_numericas(df_1, col_numericas, '0', '0', '0', '0') # imputamos variables numericas
df_1, resumen_cat_reagrupadas, dicc_cat_reagrupadas = reagrupar_categorias(df_1, col_categoricas, 'target_m') # se reagrupan categoricas
df_1 = select_features_by_correlation(df_1, col_numericas, 'target_m', 0.9, 0.9, True) # se eliminan variables altamente correlacionadas
# df, col_fechas, resumen_nuevas_col = procesar_fechas(df, col_fechas) # se crean nuevas columnas con informacion de fechas

# con el df final volvemos a correr la columnas
print("\n\033[1m\033[4mCLASIFICANDO COLUMNAS FINALES...\033[0m\n")
col_target, col_fechas, col_categoricas, col_numericas, df_1 = clasificar_columnas(df_1, 'target_m') # clasificamos columnas

PRE-PROCESANDO DF


FILTRANDO COLUMNAS...

Shape inicial del df: (1426243, 187)

Se eliminan del df las siguientes columnas:
 - codunico
 - p_codmes

Listado de columnas a conservar (no se aplica filtro segun porcentaje de datos faltates):
 - num_documento
 - nivel_riesgo_lsb_total
 - nivel_riesgo_lsb_ultima
 - fecha_mas_antiguo_lsb
 - fecha_mas_reciente_lsb
 - flag_ros_12m
 - flag_ros_hist
 - q_ros_hist
 - cantidad_lsb
 - monto_ro_debajo_umbral
 - q_ro_debajo_umbral
 - cp_cant_trx_ing_ros
 - cp_monto_total_ing_ros
 - cp_cant_trx_egr_ros
 - cp_monto_total_egr_ros
 - flag_alerta_hist
 - q_alerta_hist
 - tipo_alerta_n2
 - nivel_riesgo_lsb_total
 - nivel_riesgo_lsb_ultima
 - target_m

Se eliminan las siguientes columnas por tener valor único:
 - tipo_cliente
 - subsegmento
 - fecha_ejecucion
 - flag_desv_activa
 - flag_cce_r1
 - flag_casos_12m
 - flag_desv_hist
 - flag_alerta_12m

Se eliminan 58 columnas con más del 20.0% de nulos

Shape del df: (1426131, 119)

CREANDO TARGET...

Escenari

,columna,criterio_usado,valores_imputados
0,flag_casos_hist,0,0
1,flag_variacion_abono_monto_total_5m_1m,0,0
2,flag_variacion_efect_cargos_monto_5m_1m,0,0



IMPUTANDO NULOS PARA COLUMNAS NUMERICAS...

Resumen



,variable,criterio_usado,valores_imputados
0,pasivo_soles,0,107099
1,trx_monto_abonos_1m_total,0,172353
2,trx_monto_abonos_3m_total,0,172353
3,trx_monto_abonos_6m_total,0,172353
4,trx_monto_abonos_9m_total,0,172353
5,trx_monto_abonos_12m_total,0,172353
6,trx_monto_cargos_1m_total,0,172353
7,trx_monto_cargos_3m_total,0,172353
8,trx_monto_cargos_6m_total,0,172353
9,trx_monto_cargos_9m_total,0,172353



REAGRUPANDO VARIABLES CATEGORICAS...

Resumen



""



ELIMINANDO COLUMNAS NUMERICAS ALTAMENTE CORRELACIONADAS...

Columnas eliminadas - correlación Pearson
 - trx_monto_cargos_ratio_9m_efectivo_total
 - trx_q_abonos_promedio_6m_total
 - trx_monto_abonos_12m_max
 - trx_q_cargos_promedio_6m_total
 - trx_q_cargos_promedio_3m_total
 - trx_monto_abonos_promedio_9m_total
 - trx_monto_abonos_1m_total
 - trx_monto_abonos_12m_efectivo
 - trx_monto_cargos_9m_total
 - trx_monto_cargos_3m_max
 - trx_q_abonos_9m_efectivo
 - trx_monto_abonos_6m_max
 - trx_monto_abonos_promedio_3m_total
 - trx_q_abonos_12m_efectivo
 - trx_monto_abonos_3m_total
 - trx_q_cargos_12m_efectivo
 - trx_q_abonos_promedio_9m_total
 - trx_q_abonos_1m_total
 - trx_q_cargos_promedio_12m_total
 - cp_monto_total_egr_ros
 - trx_monto_cargos_promedio_9m_total
 - trx_monto_cargos_1m_max
 - trx_monto_cargos_9m_max
 - trx_monto_abonos_ratio_3m_efectivo_total
 - dif_monto_abonos_cargos_efectivo_6m
 - trx_monto_abonos_ratio_1m_efectivo_total
 - trx_monto_abonos_ratio_6m_efectivo_total
 - t

In [55]:
df_1.shape

(1426131, 34)

In [56]:
to_drop =  [ 'tipo_documento',
 'fecha_vinculacion'
           ]
df_2 = df_1.drop(to_drop,axis=1)

In [57]:
df_2.dtypes

num_documento                               string[python]
mes_base                                             Int32
fecha_constitucion                                  object
pasivo_soles                                       float64
trx_monto_abonos_6m_efectivo                       float64
trx_monto_cargos_6m_efectivo                       float64
trx_q_cargos_3m_total                                Int32
trx_q_abonos_promedio_3m_total                     float64
trx_monto_cargos_promedio_3m_total                 float64
trx_q_abonos_ratio_1m_efectivo_total               float64
trx_q_abonos_ratio_3m_efectivo_total               float64
trx_q_abonos_ratio_9m_efectivo_total               float64
trx_monto_cargos_ratio_1m_efectivo_total           float64
trx_monto_abonos_3m_max                            float64
edad_constitucion                                    Int32
antiguedad                                           Int32
nivel_riesgo_lsb_total                             float

In [58]:
str_vars = df_2.dtypes.index[(df_2.dtypes ==  'string[python]')]
for var in str_vars:
   print()
   print("***** ",var," ********")
   print("Cantidad de categorias distintas", len(df_2[var].unique()))
   print("Cantiddad de nulls",sum(df_2[var].isnull()) )
   print(df_2[var].value_counts().head())


*****  num_documento  ********
Cantidad de categorias distintas 191230
Cantiddad de nulls 2879
num_documento
535BD3328D645F85FC8A9E81C2CAB4C3FCEFE631EB05050FD7BAE8C6DA60BF1C    12
0C945ADA3EACE7296E32C00BF6B3E3FCC1F86CE4643636A170C47D2E42492EB8    11
64EB24C21694C3AB21270042495936E9DB96D80BB969D54621E8FA75CA29B027    11
FAE7BA7E721AC9552E87440A16F0C4A1BDDE2877596FA722CDFE6E6567E846DF    11
A7776F986F49EA5D92A8CC6F8938E77456454EFC3AA456E08108C478EE1C13E1    11
Name: count, dtype: Int64

*****  provincia  ********
Cantidad de categorias distintas 199
Cantiddad de nulls 2
provincia
LIMA        788774
TRUJILLO     73928
AREQUIPA     67794
CALLAO       45405
CUSCO        38193
Name: count, dtype: Int64

*****  departamento  ********
Cantidad de categorias distintas 28
Cantiddad de nulls 2
departamento
LIMA           810072
LA LIBERTAD     82329
AREQUIPA        71463
CUSCO           45859
CALLAO          45405
Name: count, dtype: Int64

*****  ubigeo_cd  ********
Cantidad de categorias dist

In [59]:
# Renombrar y reordenar columnas
df_2 = df_2.rename(columns={'target_m': 'target'})
df_2 =df_2[['target'] + [c for c in df_2.columns if c != 'target']]


In [60]:
df_2["tipo_alerta_n2"] =df_2["tipo_alerta_n2"].fillna(0)

In [61]:
import pandas as pd
import numpy as np

# ==========================================
# 1. Copiar DF original
# ==========================================
df_3 = df_2.copy()

# ==========================================
# 3. Columnas Int32 → rellenar NA → convertir a int64
# ==========================================
int32_cols = df_3.select_dtypes(include=["Int32"]).columns

df_3[int32_cols] = df_3[int32_cols].fillna(0).astype("int64")

# ==========================================
# 4. Columnas float → rellenar NA con 0
# ==========================================
float_cols = df_3.select_dtypes(include=["float64", "Float64"]).columns

df_3[float_cols] = df_3[float_cols].fillna(0)

# ==========================================
# 5. Columnas boolean → rellenar NA con False
# ==========================================
bool_cols = df_3.select_dtypes(include=["boolean"]).columns

df_3[bool_cols] = df_3[bool_cols].fillna(False)

# ==========================================
# 6. Columnas categóricas (strings) → NA = "SIN_INFO"
# ==========================================
cat_cols = df_3.select_dtypes(include=["object", "string"]).columns

df_3[cat_cols] = df_3[cat_cols].fillna("SIN_INFO")



In [62]:
df_3[['mes_base', 'target']].value_counts()

mes_base  target
202509    0         163493
202508    0         161800
202505    0         160642
202507    0         160046
202506    0         158656
202504    0         156594
202503    0         155689
202502    0         154616
202501    0         153419
202505    1            161
202503    1            155
202502    1            153
202504    1            129
202501    1            127
202508    1            127
202506    1            116
202509    1            105
202507    1            103
Name: count, dtype: int64

# CASO TODA LA BASE

In [63]:
#Caso 2 toda la base//con la totalidad

In [64]:
import pandas as pd
from sklearn.utils import resample

# ===========================
# 1️⃣ Separar antiguos y recientes
# ===========================
df_antiguos = df_3[df_3['mes_base'] <= 202507].copy()
df_recientes = df_3[df_3['mes_base'] == 202508].copy()  # Test completo

# ===========================
# 2️⃣ Separar clases en antiguos
# ===========================
df_antiguos_con_alerta = df_antiguos[df_antiguos['tipo_alerta_n2'] != "0"]  # Con alerta
df_antiguos_sin_alerta = df_antiguos[df_antiguos['tipo_alerta_n2'] == "0"]  # Sin alerta

# ===========================
# 3️⃣ Filtrar clases target == 1 (minoritarios)
# ===========================
df_antiguos_1 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 1]  # Alerta y target == 1
df_antiguos_0 = df_antiguos_con_alerta[df_antiguos_con_alerta['target'] == 0]  # Alerta y target == 0

# ===========================
# 4️⃣ Balanceo de clases en TRAIN
# ===========================
n_pos = len(df_antiguos_1)  # Cantidad de clases 1 (minoritarias)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # Queremos 99.5% de clase 0

# Ajustar clase 0 (target == 0)
if n_neg_deseado <= len(df_antiguos_0):
    df_antiguos_0_bal = resample(df_antiguos_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_antiguos_0_bal = resample(df_antiguos_0, replace=True, n_samples=n_neg_deseado, random_state=42)

# ===========================
# 5️⃣ Combinar con clases 1 (target == 1)
# ===========================
df_antiguos_1_bal = df_antiguos_1  # Mantener todas las clases 1 (no se ajustan, ya que se mantiene su porcentaje)

# Combinar 0 (balanceado) y 1 (original) para el dataset de entrenamiento
df_antiguos_balanceados = pd.concat([df_antiguos_0_bal, df_antiguos_1_bal], axis=0)

# ===========================
# 6️⃣ Tomamos el 0.5% de los SIN alerta
# ===========================
# Para tener el 99.5% de clase 0 y 0.5% de clase 1 en el TRAIN
porcentaje_sin_alerta = 0.005  # 0.5% de los registros sin alerta

# Tomamos el 0.5% de los casos sin alerta (sin alertas)
df_sin_alerta_sample = df_antiguos_sin_alerta.sample(frac=porcentaje_sin_alerta, random_state=42)

# ===========================
# 7️⃣ Concatenar alertas + 0.5% de sin alerta
# ===========================
df_train = pd.concat([df_antiguos_balanceados, df_sin_alerta_sample], axis=0)

# Barajamos los datos para evitar sesgo de orden
df_train = df_train.sample(frac=1, random_state=42).reset_index(drop=True)

# ===========================
# 8️⃣ Mantener df_test intacto
# ===========================
# Filtrar df_test para asegurarnos de que tiene solo los registros del test (mes_base >= 202508)
df_test = df_3[df_3['mes_base'] == 202508].copy()

# ===========================
# 9️⃣ Combinar df_train y df_test para df_7
# ===========================
df_4 = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)

# ===========================
# 10️⃣ Borrar columna 'tipo_alerta_n2' de df_7
# ===========================
#df_4 = df_4.drop(columns=['tipo_alerta_n2'], errors='ignore')

# ===========================
# 11️⃣ Verificar distribución final
# ===========================
print("Distribución final de clases en df_train (target):")
print(df_train['target'].value_counts(normalize=True))

print("\nDistribución final de tipo_alerta_n2 en df_train (Eliminada):")
print(df_train['tipo_alerta_n2'].value_counts())

print("\nDistribución final de clases en df_test (target):")
print(df_test['target'].value_counts(normalize=True))

print("\nDistribución final de clases en df_7 (target):")
print(df_4['target'].value_counts(normalize=True))

#print("\nDistribución final de tipo_alerta_n2 en df_7 (Eliminada):")
#print(df_7['tipo_alerta_n2'].value_counts())  # Debería no mostrar nada, ya que fue eliminada



Distribución final de clases en df_train (target):
target
0   0.99
1   0.01
Name: proportion, dtype: float64

Distribución final de tipo_alerta_n2 en df_train (Eliminada):
tipo_alerta_n2
0                  187275
AUTOMATICA            926
MANUAL                430
SEMI AUTOMATICA       169
Name: count, dtype: int64

Distribución final de clases en df_test (target):
target
0   1.00
1   0.00
Name: proportion, dtype: float64

Distribución final de clases en df_7 (target):
target
0   1.00
1   0.00
Name: proportion, dtype: float64


#Tomar los cero como parte del los negativos, completos!°!!!!

In [65]:
df_5 = df_4

In [66]:
# Eliminar columnas 'num_documento' y 'mes_base' si existen
cols_a_eliminar = ["fecha_constitucion"]
df_5 = df_5.drop(columns=cols_a_eliminar, errors="ignore")


In [67]:
df_5[['mes_base', 'target']].value_counts()

mes_base  target
202508    0         161800
202507    0          27648
202505    0          27288
202506    0          27157
202504    0          26652
202503    0          26526
202502    0          26405
202501    0          26180
202505    1            161
202503    1            155
202502    1            153
202504    1            129
202501    1            127
202508    1            127
202506    1            116
202507    1            103
Name: count, dtype: int64

In [68]:
df_5.tipo_alerta_n2.value_counts()

tipo_alerta_n2
0                  348629
AUTOMATICA           1358
MANUAL                511
SEMI AUTOMATICA       229
Name: count, dtype: int64

In [69]:
# Convertir explícitamente la columna problemática a string
df_5['tipo_alerta_n2'] = df_5['tipo_alerta_n2'].astype(str)

In [38]:
df_5.to_parquet(
    's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA/data_pn_total.parquet',
    index=False
)

In [70]:
df_5.head()

,target,num_documento,mes_base,pasivo_soles,trx_monto_abonos_6m_efectivo,trx_monto_cargos_6m_efectivo,trx_q_cargos_3m_total,trx_q_abonos_promedio_3m_total,trx_monto_cargos_promedio_3m_total,trx_q_abonos_ratio_1m_efectivo_total,...,ubigeo_cd,sectorista_id,ciiu_v4,flag_casos_hist,flag_variacion_abono_monto_total_5m_1m,flag_variacion_efect_cargos_monto_5m_1m,q_ro_debajo_umbral,q_alerta_hist,q_ros_hist,tipo_alerta_n2
0,0,0AA981166B1EB49EAD83A1D4CF894876897435330C75D4...,202503,51240.25,10286.00,0.00,8,1.00,0.00,0.50,...,040117,D1120,5610,0,1,1,0,0,0,0
1,0,ACCC1B5A6494708E1F3E897DF72277F579B6B6551025FC...,202506,1.51,10000.00,2240.00,78,2.33,1.33,0.00,...,150132,D1120,4290,0,0,0,0,0,0,0
2,0,67883D40C5EE87677D933F0D16E30D1F70446B8870C954...,202507,0.00,0.00,0.00,0,0.00,0.00,0.00,...,150131,D1120,4659,0,0,0,0,0,0,0
3,0,0DD47D0D10BB8A851F4A3ADFE1EC18CD0D5379C79352C4...,202505,2149.90,0.00,0.00,3,0.00,0.00,0.00,...,150110,D1120,SIN_INFO,0,0,0,0,0,0,0
4,0,F91DAC8A475A2A374DBB59664B243145377350C22AB42B...,202501,0.00,0.00,0.00,4,0.33,0.00,0.00,...,150116,D1120,4620,0,1,1,0,0,0,0


In [71]:
# Separar train/test por mes_base
df_train = df_5[df_5["mes_base"] <= 202507].copy()
df_test  = df_5[df_5["mes_base"] == 202508].copy()

# Concatenar para LabelEncoding
categoricas = ["flag_casos_hist", "flag_variacion_abono_monto_total_5m_1m",
               "flag_variacion_efect_cargos_monto_5m_1m", "provincia",
               "departamento", "ubigeo_cd", "sectorista_id", "ciiu_v4"]

df_all = pd.concat([df_train, df_test], axis=0)

for c in categoricas:
    le = LabelEncoder()
    df_all[c] = le.fit_transform(df_all[c].astype(str))

# Separar nuevamente
df_train = df_all.loc[df_all["mes_base"] <= 202507].copy()
df_test  = df_all.loc[df_all["mes_base"] == 202508].copy()

# Ahora puedes borrar columnas no deseadas
cols_drop = ["mes_base","tipo_alerta_n2","num_documento"]
df_train = df_train.drop(columns=cols_drop, errors="ignore")
df_test  = df_test.drop(columns=cols_drop, errors="ignore")




In [72]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score

# ========================================
# 1️⃣ Separar features y target
# ========================================
target = "target"
X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_test  = df_test.drop(columns=[target])
y_test  = df_test[target]

# ========================================
# 2️⃣ Columnas categóricas y numéricas
# ========================================
categoricas = X_train.select_dtypes(include=["object", "string", "bool"]).columns.tolist()
numericas   = [c for c in X_train.columns if c not in categoricas]

# ========================================
# 3️⃣ OneHot transformer
# ========================================
onehot = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categoricas)],
    remainder="passthrough"
)

# ========================================
# 4️⃣ Pipeline XGBoost
# ========================================
pipeline_xgb = Pipeline(steps=[
    ("onehot", onehot),
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        tree_method="hist",
        use_label_encoder=False,
        random_state=42
    ))
])

# ========================================
# 5️⃣ Espacio HPO
# ========================================
from scipy.stats import randint, uniform

param_dist = {
    "xgb__max_depth": randint(3, 8),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.6, 0.4),
    "xgb__colsample_bytree": uniform(0.6, 0.4),
    "xgb__gamma": uniform(0, 5),
    "xgb__min_child_weight": randint(1, 10),
    "xgb__n_estimators": randint(200, 800)
}

# ========================================
# 6️⃣ Cross-validation
# ========================================
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ========================================
# 7️⃣ RandomizedSearchCV para PRECISIÓN
# ========================================
search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=15,          # menos iteraciones para acelerar
    scoring="precision",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# ========================================
# 8️⃣ Entrenar
# ========================================
search_xgb.fit(X_train, y_train)

# ========================================
# 9️⃣ Resultados HPO
# ========================================
print("🏆 Mejores hiperparámetros para maximizar PRECISIÓN:")
print(search_xgb.best_params_)

# ========================================
# 🔟 Evaluar sobre test
# ========================================
y_pred_prob = search_xgb.predict_proba(X_test)[:,1]

# Métricas
auc = roc_auc_score(y_test, y_pred_prob)
gini = 2*auc - 1

# KS Score
def ks_score(y_true, y_score):
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df = df.sort_values("score", ascending=False)
    df["cum_event"] = (df["y"]==1).cumsum() / df["y"].sum()
    df["cum_nonevent"] = (df["y"]==0).cumsum() / (len(df) - df["y"].sum())
    ks = (df["cum_event"] - df["cum_nonevent"]).max()
    return ks

ks = ks_score(y_test, y_pred_prob)

print(f"\n📊 Métricas en TEST:")
print(f"AUC  : {auc:.4f}")
print(f"Gini : {gini:.4f}")
print(f"KS   : {ks:.4f}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the

🏆 Mejores hiperparámetros para maximizar PRECISIÓN:
{'xgb__colsample_bytree': 0.6232334448672797, 'xgb__gamma': 4.330880728874676, 'xgb__learning_rate': 0.10016725176148131, 'xgb__max_depth': 5, 'xgb__min_child_weight': 6, 'xgb__n_estimators': 508, 'xgb__subsample': 0.9879639408647978}

📊 Métricas en TEST:
AUC  : 0.9934
Gini : 0.9868
KS   : 0.9519


In [73]:
# ==============================================================
# 7) Generar deciles y micro-segmentación (CORREGIDO)
# ==============================================================

# Asegurar que df_test tenga los scores del modelo
df_eval = df_test.copy()
df_eval['score'] = y_pred_prob  # score del mejor modelo


# ==============================================================
# 7a) DECILES (10 grupos)
# ==============================================================
df_eval["decile"] = pd.qcut(df_eval["score"].rank(method="first"), 10, labels=False) + 1

tabla_deciles = df_eval.groupby("decile").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean")
).reset_index()

tabla_deciles["event_rate"] = tabla_deciles["events"] / tabla_deciles["count"]
tabla_deciles["precision"] = tabla_deciles["event_rate"]

# ORDEN CORRECTO: mayor score → menor score
tabla_deciles = tabla_deciles.sort_values("score_mean", ascending=False).reset_index(drop=True)

# RECALL ACUMULADO
tabla_deciles["recall"] = tabla_deciles["events"].cumsum() / tabla_deciles["events"].sum()

tabla_deciles["lift"] = tabla_deciles["event_rate"] / df_eval["target"].mean()

print("\n📊 TABLA POR DECILES:")
print(tabla_deciles)



# ==============================================================
# 7b) MICRO-SEGMENTACIÓN (500 grupos)
# ==============================================================
df_eval["bin_500"] = pd.qcut(df_eval["score"].rank(method="first"), 500, labels=False) + 1

tabla_500 = df_eval.groupby("bin_500").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean")
).reset_index()

tabla_500["event_rate"] = tabla_500["events"] / tabla_500["count"]
tabla_500["precision"] = tabla_500["event_rate"]

# ORDEN CORRECTO
tabla_500 = tabla_500.sort_values("score_mean", ascending=False).reset_index(drop=True)

# RECALL ACUMULADO
tabla_500["recall"] = tabla_500["events"].cumsum() / tabla_500["events"].sum()

tabla_500["lift"] = tabla_500["event_rate"] / df_eval["target"].mean()

print("\n📊 PRIMEROS 10 GRUPOS (500 bins):")
print(tabla_500.head(10))



# ==============================================================
# 7c) MICRO-SEGMENTACIÓN (1000 grupos)
# ==============================================================
df_eval["bin_1000"] = pd.qcut(df_eval["score"].rank(method="first"), 1000, labels=False) + 1

tabla_1000 = df_eval.groupby("bin_1000").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean")
).reset_index()

tabla_1000["event_rate"] = tabla_1000["events"] / tabla_1000["count"]
tabla_1000["precision"] = tabla_1000["event_rate"]

# ORDEN CORRECTO
tabla_1000 = tabla_1000.sort_values("score_mean", ascending=False).reset_index(drop=True)

# RECALL ACUMULADO
tabla_1000["recall"] = tabla_1000["events"].cumsum() / tabla_1000["events"].sum()

tabla_1000["lift"] = tabla_1000["event_rate"] / df_eval["target"].mean()

print("\n📊 PRIMEROS 10 GRUPOS (1000 bins):")
print(tabla_1000.head(10))


📊 TABLA POR DECILES:
   decile  count  events  score_mean  event_rate  precision  recall  lift
0      10  16193     127        0.04        0.01       0.01    1.00 10.00
1       9  16193       0        0.00        0.00       0.00    1.00  0.00
2       8  16192       0        0.00        0.00       0.00    1.00  0.00
3       7  16193       0        0.00        0.00       0.00    1.00  0.00
4       6  16192       0        0.00        0.00       0.00    1.00  0.00
5       5  16193       0        0.00        0.00       0.00    1.00  0.00
6       4  16193       0        0.00        0.00       0.00    1.00  0.00
7       3  16192       0        0.00        0.00       0.00    1.00  0.00
8       2  16193       0        0.00        0.00       0.00    1.00  0.00
9       1  16193       0        0.00        0.00       0.00    1.00  0.00

📊 PRIMEROS 10 GRUPOS (500 bins):
   bin_500  count  events  score_mean  event_rate  precision  recall   lift
0      500    324      69        0.64        0.21     

In [74]:
# ==============================================================
# 1) TRAIN / TEST + GUARDAR COLUMNAS CLAVE DESDE EL PRINCIPIO
# ==============================================================
df_train = df_5[df_5["mes_base"] <= 202507].copy()
df_test  = df_5[df_5["mes_base"] == 202508].copy()

# GUARDAMOS desde ya las columnas que vamos a necesitar después
df_test_ids_y_alertas = df_test[["num_documento", "tipo_alerta_n2", "target"]].copy()

# Label encoding de categóricas
categoricas = ["flag_casos_hist", "flag_variacion_abono_monto_total_5m_1m",
               "flag_variacion_efect_cargos_monto_5m_1m", "provincia",
               "departamento", "ubigeo_cd", "sectorista_id", "ciiu_v4"]

df_all = pd.concat([df_train, df_test])
for c in categoricas:
    le = LabelEncoder()
    df_all[c] = le.fit_transform(df_all[c].astype(str))

df_train = df_all[df_all["mes_base"] <= 202507].copy()
df_test  = df_all[df_all["mes_base"] == 202508].copy()

# Ahora SÍ borramos todo lo que no sirve para entrenar
df_train = df_train.drop(columns=["mes_base", "num_documento", "tipo_alerta_n2"], errors="ignore")
df_test  = df_test.drop(columns=["mes_base", "num_documento", "tipo_alerta_n2"], errors="ignore")

In [75]:
# ==============================================================
# RECUPERAR ALERTAS + TABLAS CON % DE CLIENTES QUE TUVIERON ALERTA
# ==============================================================
# Reconstruimos df_eval con los scores y el índice original
df_eval = df_test.copy()
df_eval["score"] = y_pred_prob

# Agregamos un índice temporal para hacer merge perfecto
df_eval = df_eval.reset_index(drop=True)
df_test_ids_y_alertas = df_test_ids_y_alertas.reset_index(drop=True)

# Merge usando el orden original (índice) → nunca falla
df_eval = pd.concat([df_eval, df_test_ids_y_alertas[["num_documento", "tipo_alerta_n2"]]], axis=1)

# Cliente tuvo alerta?
df_eval["tuvo_alerta"] = (
    df_eval["tipo_alerta_n2"].notna() & 
    (df_eval["tipo_alerta_n2"].astype(str).str.strip() != "") & 
    (df_eval["tipo_alerta_n2"].astype(str) != "0")
)

# ==============================================================
# FUNCIÓN MÁGICA PARA TODAS LAS TABLAS
# ==============================================================
def tabla_con_pct_alerta(df, n_bins, nombre_bin):
    df = df.copy()
    df[nombre_bin] = pd.qcut(df["score"].rank(method="first"), n_bins, labels=False, duplicates="drop") + 1
    tabla = df.groupby(nombre_bin).agg(
        clientes=("target", "size"),
        eventos=("target", "sum"),
        score_prom=("score", "mean"),
        pct_con_alerta=("tuvo_alerta", "mean")
    ).reset_index()
    tabla["event_rate"] = tabla["eventos"] / tabla["clientes"]
    tabla = tabla.sort_values("score_prom", ascending=False).reset_index(drop=True)
    total_eventos = tabla["eventos"].sum()
    tabla["recall_acum"] = tabla["eventos"].cumsum() / total_eventos
    tasa_global = df["target"].mean()
    tabla["lift"] = tabla["event_rate"] / tasa_global
    tabla["pct_con_alerta"] = (tabla["pct_con_alerta"] * 100).round(1)
    return tabla

# GENERAR LAS 3 TABLAS
print("\n" + "="*90)
print("DECILES + % CLIENTES CON ALERTA")
tabla_deciles = tabla_con_pct_alerta(df_eval, 10, "decile")
print(tabla_deciles[["decile","clientes","eventos","score_prom","event_rate","pct_con_alerta","recall_acum","lift"]].round(4))



print("\n" + "="*90)
print("TOP 20 DE 1000 BINS + % CON ALERTA")
tabla_1000 = tabla_con_pct_alerta(df_eval, 1000, "bin1000")
print(tabla_1000.head(20)[["bin1000","clientes","eventos","score_prom","event_rate","pct_con_alerta","recall_acum","lift"]].round(4))

# Exportar todo
#with pd.ExcelWriter("resultados_FINAL_con_pct_alertas.xlsx") as writer:
#    tabla_deciles.to_excel(writer, sheet_name="Deciles", index=False)
#    tabla_500.head(50).to_excel(writer, sheet_name="Top50_500", index=False)
#    tabla_1000.head(50).to_excel(writer, sheet_name="Top50_1000", index=False)

#print("\n¡TODO LISTO! Archivo Excel generado: resultados_FINAL_con_pct_alertas.xlsx")
print("="*90)


DECILES + % CLIENTES CON ALERTA
   decile  clientes  eventos  score_prom  event_rate  pct_con_alerta  \
0      10     16193      127        0.04        0.01            3.50   
1       9     16193        0        0.00        0.00            0.00   
2       8     16192        0        0.00        0.00            0.00   
3       7     16193        0        0.00        0.00            0.00   
4       6     16192        0        0.00        0.00            0.00   
5       5     16193        0        0.00        0.00            0.00   
6       4     16193        0        0.00        0.00            0.00   
7       3     16192        0        0.00        0.00            0.00   
8       2     16193        0        0.00        0.00            0.00   
9       1     16193        0        0.00        0.00            0.00   

   recall_acum  lift  
0         1.00 10.00  
1         1.00  0.00  
2         1.00  0.00  
3         1.00  0.00  
4         1.00  0.00  
5         1.00  0.00  
6         1.0

In [76]:
# ======================================================
# FUNCIÓN GENERAL PARA LAS 3 TABLAS
# ======================================================

def tabla_final(df, n_bins, nombre_bin):
    df = df.copy()

    # Crear bins
    df[nombre_bin] = pd.qcut(
        df["score"].rank(method="first"),
        n_bins,
        labels=False,
        duplicates="drop"
    ) + 1

    # -------------------------------------------
    # Agrupar
    # -------------------------------------------
    tabla = df.groupby(nombre_bin).agg(
        cantidad_total_casos=("target", "size"),
        score_min=("score", "min"),
        score_max=("score", "max"),
        casos_positivos=("target", "sum"),
        casos_con_alerta=("tuvo_alerta", "sum"),
    ).reset_index()

    # Casos sin alerta
    tabla["casos_sin_alerta"] = tabla["cantidad_total_casos"] - tabla["casos_con_alerta"]

    # -------------------------------------------
    # ORDENAR POR SCORE DESCENDENTE (IMPORTANTE)
    # -------------------------------------------
    tabla = tabla.sort_values("score_max", ascending=False).reset_index(drop=True)

    # -------------------------------------------
    # PRECISIÓN SOLO SOBRE CASOS CON ALERTA
    # -------------------------------------------
    # positivos dentro de alertas por grupo
    positivos_con_alerta = (
        df[df["tuvo_alerta"] == True]
        .groupby(df[nombre_bin])["target"]
        .sum()
    )

    tabla["precision"] = tabla[nombre_bin].map(positivos_con_alerta) / tabla["casos_con_alerta"]
    tabla["precision"] = tabla["precision"].fillna(0)

    # -------------------------------------------
    # RECALL ACUMULADO
    # -------------------------------------------
    total_eventos = tabla["casos_positivos"].sum()
    tabla["recall"] = tabla["casos_positivos"].cumsum() / total_eventos

    # -------------------------------------------
    # Rango del score
    # -------------------------------------------
    tabla["rango_probabilidad"] = tabla.apply(
        lambda x: f"{x['score_min']:.3f} - {x['score_max']:.3f}", axis=1
    )

    # Columnas finales
    tabla = tabla[[
        nombre_bin,
        "cantidad_total_casos",
        "rango_probabilidad",
        "casos_positivos",
        "casos_con_alerta",
        "precision",
        "recall",
        "casos_sin_alerta"
    ]]

    return tabla


# ======================================================
# TABLA 1 — 10 GRUPOS (TODOS LOS DATOS)
# ======================================================
print("\n" + "="*90)
print("TABLA 1 — 10 GRUPOS (TODOS LOS DATOS)")

tabla10 = tabla_final(df_eval, 10, "grupo10")
print(tabla10)


# ======================================================
# TABLA 2 — 1000 GRUPOS (TODOS LOS DATOS)
# ======================================================
print("\n" + "="*90)
print("TABLA 2 — 1000 GRUPOS (TODOS LOS DATOS)")

tabla1000 = tabla_final(df_eval, 1000, "grupo1000")
print(tabla1000.head(20))


# ======================================================
# TABLA 3 — APLICANDO FILTRO SCORE > 0.5
# ======================================================
df_eval_sc05 = df_eval[df_eval["score"] > 0.5].copy()

print("\n" + "="*90)
print("TABLA 3 — SCORE > 0.5 (10 GRUPOS)")

tabla10_sc05 = tabla_final(df_eval_sc05, 10, "grupo10_sc05")
print(tabla10_sc05)





TABLA 1 — 10 GRUPOS (TODOS LOS DATOS)
   grupo10  cantidad_total_casos rango_probabilidad  casos_positivos  \
0       10                 16193      0.001 - 0.928              127   
1        9                 16193      0.000 - 0.001                0   
2        8                 16192      0.000 - 0.000                0   
3        7                 16193      0.000 - 0.000                0   
4        6                 16192      0.000 - 0.000                0   
5        5                 16193      0.000 - 0.000                0   
6        4                 16193      0.000 - 0.000                0   
7        3                 16192      0.000 - 0.000                0   
8        2                 16193      0.000 - 0.000                0   
9        1                 16193      0.000 - 0.000                0   

   casos_con_alerta  precision  recall  casos_sin_alerta  
0               573       0.22    1.00             15620  
1                 0       0.00    1.00            

In [77]:
!pip install openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [openpyxl]1/2 [openpyxl]


In [78]:
with pd.ExcelWriter("tablas_resultado.xlsx") as writer:
    tabla10.to_excel(writer, sheet_name="10_grupos", index=False)
    tabla1000.to_excel(writer, sheet_name="1000_grupos", index=False)
    tabla10_sc05.to_excel(writer, sheet_name="10_grupos_score>0.5", index=False)

print("Archivo Excel generado: tablas_resultado.xlsx")


Archivo Excel generado: tablas_resultado.xlsx


In [79]:
def tabla_final(df, n_bins, nombre_bin):
    df = df.copy()

    # Crear bins
    df[nombre_bin] = pd.qcut(
        df["score"].rank(method="first"),
        n_bins,
        labels=False,
        duplicates="drop"
    ) + 1

    # -------------------------------------------
    # Grupales base
    # -------------------------------------------
    tabla = df.groupby(nombre_bin).agg(
        cantidad_total_casos=("target", "size"),
        score_min=("score", "min"),
        score_max=("score", "max"),
        casos_positivos=("target", "sum"),
        casos_con_alerta=("tuvo_alerta", "sum"),
    ).reset_index()

    # Casos sin alerta
    tabla["casos_sin_alerta"] = tabla["cantidad_total_casos"] - tabla["casos_con_alerta"]

    # -------------------------------------------
    # Precisión solo entre casos con alerta
    # -------------------------------------------
    positivos_con_alerta = (
        df[df["tuvo_alerta"] == True]
        .groupby(df[nombre_bin])["target"]
        .sum()
    )

    tabla["precision"] = tabla[nombre_bin].map(positivos_con_alerta) / tabla["casos_con_alerta"]
    tabla["precision"] = tabla["precision"].fillna(0)  # evitar división por cero

    # -------------------------------------------
    # Rango del score
    # -------------------------------------------
    tabla["rango_probabilidad"] = tabla.apply(
        lambda x: f"{x['score_min']:.3f} - {x['score_max']:.3f}", axis=1
    )

    # -------------------------------------------
    # Orden descendente por score (IMPORTANTE para acumulados)
    # -------------------------------------------
    tabla = tabla.sort_values("score_max", ascending=False).reset_index(drop=True)

    # -------------------------------------------
    # ACUMULADOS TRADICIONALES DESCENDENTES
    # -------------------------------------------
    tabla["cantidad_total_casos_acum"] = tabla["cantidad_total_casos"].cumsum()
    tabla["casos_positivos_acum"] = tabla["casos_positivos"].cumsum()
    tabla["casos_con_alerta_acum"] = tabla["casos_con_alerta"].cumsum()
    tabla["casos_sin_alerta_acum"] = tabla["casos_sin_alerta"].cumsum()

    # Recall acumulado
    total_eventos = tabla["casos_positivos"].sum()
    tabla["recall"] = tabla["casos_positivos_acum"] / total_eventos

    # -------------------------------------------
    # Orden final solicitado de columnas
    # -------------------------------------------
    tabla = tabla[[
        nombre_bin,
        "cantidad_total_casos",
        "cantidad_total_casos_acum",
        "rango_probabilidad",
        "casos_positivos",
        "casos_positivos_acum",
        "casos_con_alerta",
        "casos_con_alerta_acum",
        "precision",
        "recall",
        "casos_sin_alerta",
        "casos_sin_alerta_acum",
    ]]

    return tabla


In [80]:
# TABLA 1 — 10 GRUPOS
tabla10 = tabla_final(df_eval, 10, "grupo10")
print(tabla10)

# TABLA 2 — 1000 GRUPOS
tabla1000 = tabla_final(df_eval, 1000, "grupo1000")
print(tabla1000.head(20))

# TABLA 3 — SCORE > 0.5
df_eval_sc05 = df_eval[df_eval["score"] > 0.5].copy()

tabla10_sc05 = tabla_final(df_eval_sc05, 10, "grupo10_sc05")
print(tabla10_sc05)

tabla1000_sc05 = tabla_final(df_eval_sc05, 1000, "grupo1000_sc05")
print(tabla1000_sc05.head(20))


   grupo10  cantidad_total_casos  cantidad_total_casos_acum  \
0       10                 16193                      16193   
1        9                 16193                      32386   
2        8                 16192                      48578   
3        7                 16193                      64771   
4        6                 16192                      80963   
5        5                 16193                      97156   
6        4                 16193                     113349   
7        3                 16192                     129541   
8        2                 16193                     145734   
9        1                 16193                     161927   

  rango_probabilidad  casos_positivos  casos_positivos_acum  casos_con_alerta  \
0      0.001 - 0.928              127                   127               573   
1      0.000 - 0.001                0                   127                 0   
2      0.000 - 0.000                0                   127    

In [81]:
# ==============================================================
# TABLA AGRUPADA DE 1000 BINS – VERSIÓN DEFINITIVA (la que vas a presentar)
# ==============================================================

# Aseguramos que tabla_1000 está ordenada de mayor a menor riesgo
tabla_1000 = tabla_1000.sort_values("score_prom", ascending=False).reset_index(drop=True)

# Definimos los cortes típicos que le encantan a gerencia y SBS
cortes = [1, 2, 3, 5, 10, 20, 30, 50, 100, 200, 300, 500, 1000]

tabla_agrupada = []

for k in cortes:
    temp = tabla_1000.head(k).agg({
        'clientes': 'sum',
        'eventos': 'sum',
        'pct_con_alerta': 'mean'  # promedio ponderado no, pero para top pequeños da casi igual y queda lindo
    })
    
    clientes_acum = temp["clientes"]
    eventos_acum = temp["eventos"]
    pct_alerta_prom = temp["pct_con_alerta"]
    
    tabla_agrupada.append({
        'Top_k_grupos': f"Top {k}",
        'Clientes_acum': int(clientes_acum),
        'Eventos_capturados': int(eventos_acum),
        '%_Riesgo_capturado': round(eventos_acum / tabla_1000["eventos"].sum() * 100, 1),
        '%_Clientes_usados': round(clientes_acum / tabla_1000["clientes"].sum() * 100, 2),
        'Lift_acumulado': round((eventos_acum / clientes_acum) / (tabla_1000["eventos"].sum() / tabla_1000["clientes"].sum()), 2),
        '%_con_Alerta_prom': round(pct_alerta_prom, 1)
    })

# Convertimos a DataFrame lindo
tabla_final = pd.DataFrame(tabla_agrupada)

# Mostramos lindo en pantalla
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
print("\n" + "="*100)
print("TABLA AGRUPADA 1000 BINS – CONCENTRACIÓN DEL RIESGO + % ALERTAS")
print("="*100)
print(tabla_final.to_string(index=False, float_format=lambda x: f"{x:.1f}"))

# Exportamos a Excel (esta hoja va a volar cabezas en la reunión
#tabla_final.to_excel("CONCENTRACION_TOP_1000_BINS_con_alertas.xlsx", index=False)
#print("\nArchivo exportado: CONCENTRACION_TOP_1000_BINS_con_alertas.xlsx")
print("="*100)


TABLA AGRUPADA 1000 BINS – CONCENTRACIÓN DEL RIESGO + % ALERTAS
Top_k_grupos  Clientes_acum  Eventos_capturados  %_Riesgo_capturado  %_Clientes_usados  Lift_acumulado  %_con_Alerta_prom
       Top 1            162                  49                38.6                0.1           385.6               40.1
       Top 2            324                  69                54.3                0.2           271.5               34.6
       Top 3            486                  78                61.4                0.3           204.6               29.8
       Top 5            810                  91                71.7                0.5           143.2               24.4
      Top 10           1620                 104                81.9                1.0            81.8               17.5
      Top 20           3239                 112                88.2                2.0            44.1               12.5
      Top 30           4858                 115                90.6              

In [82]:
# ==============================================================
# RECUPERAR tipo_alerta_n2 desde df_5 original (porque lo borraste antes)
# ==============================================================
# Volvemos a traer la columna tipo_alerta_n2 y num_documento al df_eval
df_eval = df_eval.reset_index(drop=True)
df_test_original = df_5[df_5["mes_base"] == 202508].copy()

# Aseguramos mismo orden y merge seguro
df_eval = df_eval.merge(
    df_test_original[["num_documento", "tipo_alerta_n2"]],
    on="num_documento",
    how="left"
)

# Definir si tuvo alerta o no (cualquier valor distinto de NaN, vacío o "0")
df_eval["tuvo_alerta"] = df_eval["tipo_alerta_n2"].notna() & (df_eval["tipo_alerta_n2"].astype(str).str.strip() != "") & (df_eval["tipo_alerta_n2"].astype(str) != "0")

# ==============================================================
# 7) TABLAS CON % DE CLIENTES QUE TUVIERON ALERTA
# ==============================================================

# ------------------ DECILES (10 grupos) ------------------
df_eval["decile"] = pd.qcut(df_eval["score"].rank(method="first"), 10, labels=False) + 1
tabla_deciles = df_eval.groupby("decile").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean"),
    pct_con_alerta=("tuvo_alerta", "mean")  # ← NUEVA COLUMNA
).reset_index()

tabla_deciles["event_rate"] = tabla_deciles["events"] / tabla_deciles["count"]
tabla_deciles["precision"] = tabla_deciles["event_rate"]
tabla_deciles = tabla_deciles.sort_values("score_mean", ascending=False).reset_index(drop=True)
tabla_deciles["recall"] = tabla_deciles["events"].cumsum() / tabla_deciles["events"].sum()
tabla_deciles["lift"] = tabla_deciles["event_rate"] / df_eval["target"].mean()

# Formateo bonito
tabla_deciles["pct_con_alerta"] = (tabla_deciles["pct_con_alerta"] * 100).round(1).astype(str) + "%"

print("\nTABLA POR DECILES + % CON ALERTA:")
print(tabla_deciles[[
    'decile', 'count', 'events', 'score_mean', 'event_rate',
    'pct_con_alerta', 'recall', 'lift'
]].round(4))


# ------------------ 500 bins ------------------
df_eval["bin_500"] = pd.qcut(df_eval["score"].rank(method="first"), 500, labels=False, duplicates="drop") + 1
tabla_500 = df_eval.groupby("bin_500").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean"),
    pct_con_alerta=("tuvo_alerta", "mean")
).reset_index()

tabla_500["event_rate"] = tabla_500["events"] / tabla_500["count"]
tabla_500 = tabla_500.sort_values("score_mean", ascending=False).reset_index(drop=True)
tabla_500["recall"] = tabla_500["events"].cumsum() / tabla_500["events"].sum()
tabla_500["lift"] = tabla_500["event_rate"] / df_eval["target"].mean()
tabla_500["pct_con_alerta"] = (tabla_500["pct_con_alerta"] * 100).round(1).astype(str) + "%"

print("\nPRIMEROS 15 GRUPOS (500 bins) + % CON ALERTA:")
print(tabla_500.head(15)[[
    'bin_500', 'count', 'events', 'score_mean', 'event_rate',
    'pct_con_alerta', 'recall', 'lift'
]].round(4))


# ------------------ 1000 bins ------------------
df_eval["bin_1000"] = pd.qcut(df_eval["score"].rank(method="first"), 1000, labels=False, duplicates="drop") + 1
tabla_1000 = df_eval.groupby("bin_1000").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean"),
    pct_con_alerta=("tuvo_alerta", "mean")
).reset_index()

tabla_1000["event_rate"] = tabla_1000["events"] / tabla_1000["count"]
tabla_1000 = tabla_1000.sort_values("score_mean", ascending=False).reset_index(drop=True)
tabla_1000["recall"] = tabla_1000["events"].cumsum() / tabla_1000["events"].sum()
tabla_1000["lift"] = tabla_1000["event_rate"] / df_eval["target"].mean()
tabla_1000["pct_con_alerta"] = (tabla_1000["pct_con_alerta"] * 100).round(1).astype(str) + "%"

print("\nPRIMEROS 15 GRUPOS (1000 bins) + % CON ALERTA:")
print(tabla_1000.head(15)[[
    'bin_1000', 'count', 'events', 'score_mean', 'event_rate',
    'pct_con_alerta', 'recall', 'lift'
]].round(4))



KeyError: 'tipo_alerta_n2'

[CV] END xgb__colsample_bytree=0.749816047538945, xgb__gamma=4.75357153204958, xgb__learning_rate=0.11979909127171076, xgb__max_depth=7, xgb__min_child_weight=5, xgb__n_estimators=321, xgb__subsample=0.662397808134481; total time=   5.1s
[CV] END xgb__colsample_bytree=0.6232334448672797, xgb__gamma=4.330880728874676, xgb__learning_rate=0.10016725176148131, xgb__max_depth=5, xgb__min_child_weight=6, xgb__n_estimators=508, xgb__subsample=0.9879639408647978; total time=   6.3s
[CV] END xgb__colsample_bytree=0.9329770563201687, xgb__gamma=1.0616955533913808, xgb__learning_rate=0.03727374508106509, xgb__max_depth=7, xgb__min_child_weight=1, xgb__n_estimators=659, xgb__subsample=0.8446612641953124; total time=  12.5s
[CV] END xgb__colsample_bytree=0.786705157299192, xgb__gamma=4.299702033681603, xgb__learning_rate=0.11204613078816694, xgb__max_depth=3, xgb__min_child_weight=7, xgb__n_estimators=473, xgb__subsample=0.9795542149013333; total time=   5.9s
[CV] END xgb__colsample_bytree=0.786705

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# -------------------------------
# 1️⃣ Matriz con umbral 0.5
# -------------------------------
umbral_default = 0.5
y_pred_default = (y_pred_prob >= umbral_default).astype(int)

cm_default = confusion_matrix(y_test, y_pred_default)
disp_default = ConfusionMatrixDisplay(confusion_matrix=cm_default)
print("\n📌 Matriz de Confusión (umbral 0.5)")
disp_default.plot()


In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score

# -------------------------------
# 1️⃣ Matriz con umbral 0.3
# -------------------------------
umbral_default = 0.80
y_pred_default = (y_pred_prob >= umbral_default).astype(int)

# Matriz de confusión
cm_default = confusion_matrix(y_test, y_pred_default)
disp_default = ConfusionMatrixDisplay(confusion_matrix=cm_default)
print(f"\n📌 Matriz de Confusión (umbral {umbral_default})")
disp_default.plot()

# -------------------------------
# 2️⃣ Precision y Recall
# -------------------------------
precision = precision_score(y_test, y_pred_default)
recall = recall_score(y_test, y_pred_default)

print(f"\n🎯 Precision: {precision:.4f}")
print(f"📈 Recall: {recall:.4f}")





[CV] END xgb__colsample_bytree=0.749816047538945, xgb__gamma=4.75357153204958, xgb__learning_rate=0.11979909127171076, xgb__max_depth=7, xgb__min_child_weight=5, xgb__n_estimators=321, xgb__subsample=0.662397808134481; total time=   5.1s
[CV] END xgb__colsample_bytree=0.6232334448672797, xgb__gamma=4.330880728874676, xgb__learning_rate=0.10016725176148131, xgb__max_depth=5, xgb__min_child_weight=6, xgb__n_estimators=508, xgb__subsample=0.9879639408647978; total time=   6.9s
[CV] END xgb__colsample_bytree=0.9329770563201687, xgb__gamma=1.0616955533913808, xgb__learning_rate=0.03727374508106509, xgb__max_depth=7, xgb__min_child_weight=1, xgb__n_estimators=659, xgb__subsample=0.8446612641953124; total time=  15.0s
[CV] END xgb__colsample_bytree=0.786705157299192, xgb__gamma=4.299702033681603, xgb__learning_rate=0.11204613078816694, xgb__max_depth=3, xgb__min_child_weight=7, xgb__n_estimators=473, xgb__subsample=0.9795542149013333; total time=   7.1s
[CV] END xgb__colsample_bytree=0.786705

# SOLO TRAIN ALERTAS

In [83]:
import pandas as pd
from sklearn.utils import resample

# ===========================
# 1️⃣ Filtrar SOLO alertas
# ===========================
df_alertas = df_3[df_3['tipo_alerta_n2'] != 0].copy()

# ===========================
# 2️⃣ Separar TRAIN y TEST
# ===========================
df_antiguos  = df_alertas[df_alertas['mes_base'] < 202508].copy()   # TRAIN
df_recientes = df_alertas[df_alertas['mes_base'] == 202508].copy()  # TEST

print("Registros TRAIN (solo alertas):", len(df_antiguos))
print("Registros TEST  (solo alertas):", len(df_recientes))

# ===========================
# 3️⃣ Balancear clases SOLO en TRAIN
# ===========================
df_0 = df_antiguos[df_antiguos['target'] == 0]
df_1 = df_antiguos[df_antiguos['target'] == 1]

# Tamaños deseados (5% positivos / 95% negativos)
n_pos = len(df_1)
n_neg_deseado = int(n_pos * (95 / 5))

# Balancear negativos
df_0_bal = resample(
    df_0,
    replace=(n_neg_deseado > len(df_0)),
    n_samples=n_neg_deseado,
    random_state=42
)

# Balancear positivos si falta
n_pos_deseado = int(len(df_0_bal) * (5 / 95))
df_1_bal = resample(
    df_1,
    replace=True,
    n_samples=n_pos_deseado,
    random_state=42
)

# TRAIN final
df_train_final = pd.concat([df_0_bal, df_1_bal], axis=0)
df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)

# ===========================
# 4️⃣ Combinar TRAIN + TEST
# ===========================
df_4 = pd.concat([df_train_final, df_recientes], axis=0).reset_index(drop=True)

# ===========================
# 5️⃣ Verificar
# ===========================
print("\n📌 Distribución final SOLO ALERTAS:")
print(df_4.groupby(['mes_base','target']).size())
print("\nProporción target total:")
print(df_4['target'].value_counts(normalize=True))


Registros TRAIN (solo alertas): 4419
Registros TEST  (solo alertas): 573

📌 Distribución final SOLO ALERTAS:
mes_base  target
202501    0         2484
          1          125
202502    0         2991
          1          148
202503    0         3062
          1          148
202504    0         1954
          1          151
202505    0         2642
          1          159
202506    0         2425
          1          117
202507    0         2378
          1           96
202508    0          446
          1          127
dtype: int64

Proporción target total:
target
0   0.94
1   0.06
Name: proportion, dtype: float64


In [84]:
import pandas as pd
from sklearn.utils import resample

# ===========================
# 1️⃣ Separar antiguos y recientes
# ===========================
#df_antiguos = df_6[df_6['mes_base'] < 202508].copy()
df_antiguos = df_3[(df_3['mes_base'] < 202508) & (df_3['tipo_alerta_n2'] != 0)].copy()
df_recientes = df_3[(df_3['mes_base'] == 202508) & (df_3['tipo_alerta_n2'] != 0)].copy()  # test completo
#df_recientes = df_6[df_6['mes_base'] >= 202508].copy()  # test completo

# ===========================
# 2️⃣ Separar clases en antiguos
# ===========================
df_antiguos_0 = df_antiguos[df_antiguos['target'] == 0]
df_antiguos_1 = df_antiguos[df_antiguos['target'] == 1]

# ===========================
# 3️⃣ Balancear clases: 5% minoritaria / 95% mayoritaria
# ===========================
n_pos = len(df_antiguos_1)
n_neg_deseado = int(n_pos * (95 / 5))  # cantidad deseada de clase 0

# Ajustar clase 0
if n_neg_deseado <= len(df_antiguos_0):
    df_antiguos_0_bal = resample(df_antiguos_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_antiguos_0_bal = resample(df_antiguos_0, replace=True, n_samples=n_neg_deseado, random_state=42)

# Ajustar clase 1 si hace falta
n_pos_deseado = int(len(df_antiguos_0_bal) * (5 / 95))
if n_pos_deseado > n_pos:
    df_antiguos_1_bal = resample(df_antiguos_1, replace=True, n_samples=n_pos_deseado, random_state=42)
else:
    df_antiguos_1_bal = df_antiguos_1

# ===========================
# 4️⃣ Combinar y barajar meses antiguos balanceados
# ===========================
df_antiguos_balanceados = pd.concat([df_antiguos_0_bal, df_antiguos_1_bal], axis=0)
df_antiguos_balanceados = df_antiguos_balanceados.sample(frac=1, random_state=42).reset_index(drop=True)

# ===========================
# 5️⃣ Combinar con meses recientes (test completo)
# ===========================
df_4= pd.concat([df_antiguos_balanceados, df_recientes], axis=0).reset_index(drop=True)

# ===========================
# 6️⃣ Verificar distribución
# ===========================
print("Distribución final de clases en df_7:")
print(df_4['target'].value_counts(normalize=True))

# df_7 ya contiene:
# - Meses antiguos balanceados
# - Meses recientes completos

Distribución final de clases en df_7:
target
0   0.94
1   0.06
Name: proportion, dtype: float64


In [85]:
df_4[['mes_base', 'target']].value_counts()

mes_base  target
202503    0         3062
202502    0         2991
202505    0         2642
202501    0         2484
202506    0         2425
202507    0         2378
202504    0         1954
202508    0          446
202505    1          161
202503    1          155
202502    1          153
202504    1          129
202501    1          127
202508    1          127
202506    1          116
202507    1          103
Name: count, dtype: int64

In [86]:
# Eliminar columnas 'num_documento' y 'mes_base' si existen
cols_a_eliminar = ["fecha_constitucion"]
df_4 = df_4.drop(columns=cols_a_eliminar, errors="ignore")

[CV] END xgb__colsample_bytree=0.749816047538945, xgb__gamma=4.75357153204958, xgb__learning_rate=0.11979909127171076, xgb__max_depth=7, xgb__min_child_weight=5, xgb__n_estimators=321, xgb__subsample=0.662397808134481; total time=   5.1s
[CV] END xgb__colsample_bytree=0.6232334448672797, xgb__gamma=4.330880728874676, xgb__learning_rate=0.10016725176148131, xgb__max_depth=5, xgb__min_child_weight=6, xgb__n_estimators=508, xgb__subsample=0.9879639408647978; total time=   6.3s
[CV] END xgb__colsample_bytree=0.6028265220878869, xgb__gamma=0.11531212520707879, xgb__learning_rate=0.08871619903875837, xgb__max_depth=4, xgb__min_child_weight=3, xgb__n_estimators=766, xgb__subsample=0.9932923543227152; total time=  13.7s
[CV] END xgb__colsample_bytree=0.786705157299192, xgb__gamma=4.299702033681603, xgb__learning_rate=0.11204613078816694, xgb__max_depth=3, xgb__min_child_weight=7, xgb__n_estimators=473, xgb__subsample=0.9795542149013333; total time=   5.9s
[CV] END xgb__colsample_bytree=0.98625

In [87]:
#import pandas as pd
#from sklearn.preprocessing import LabelEncoder


#to_drop =  [ 'num_documento']
#df_5 = df_4.drop(to_drop,axis=1)

# 1. Identificar columnas categóricas
#categoricas = list(df_5.select_dtypes(include=["string", "boolean"]).columns)

# 2. Aplicar LabelEncoder a cada columna
#for col in categoricas:
#    le = LabelEncoder()
#    df_5[col] = le.fit_transform(df_5[col].astype(str))

# 3. Convertir a float32 (SageMaker XGBoost lo prefiere)
#df_5 = df_5.astype("float32")

# 4. mover target a la primera columna
#target = "target"
#cols = [target] + [c for c in df_5.columns if c != target]
#df_5 = df_5[cols]

# 5. Guardar dataset para SageMaker
#df_4.to_csv("dataset_sagemaker_alertas.csv", index=False, header=False)

In [88]:
# Separar train/test por mes_base
from sklearn.preprocessing import LabelEncoder
df_train = df_4[df_4["mes_base"] < 202506].copy()
df_test  = df_4[df_4["mes_base"] >= 202506].copy()

# Concatenar para LabelEncoding
categoricas = ["flag_casos_hist", "flag_variacion_abono_monto_total_5m_1m",
               "flag_variacion_efect_cargos_monto_5m_1m", "provincia",
               "departamento", "ubigeo_cd", "sectorista_id", "ciiu_v4","tipo_alerta_n2"]

df_all = pd.concat([df_train, df_test], axis=0)

for c in categoricas:
    le = LabelEncoder()
    df_all[c] = le.fit_transform(df_all[c].astype(str))

# Separar nuevamente
df_train = df_all.loc[df_all["mes_base"] < 202508].copy()
df_test  = df_all.loc[df_all["mes_base"] == 202508].copy()

# Ahora puedes borrar columnas no deseadas
cols_drop = ["mes_base","num_documento"]
df_train = df_train.drop(columns=cols_drop, errors="ignore")
df_test  = df_test.drop(columns=cols_drop, errors="ignore")

In [89]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score, precision_score

# ========================================
# 1️⃣ Separar features y target
# ========================================
target = "target"
X_train = df_train.drop(columns=[target])
y_train = df_train[target]

X_test  = df_test.drop(columns=[target])
y_test  = df_test[target]

# ========================================
# 2️⃣ Columnas categóricas y numéricas
# ========================================
categoricas = X_train.select_dtypes(include=["object", "string", "bool"]).columns.tolist()
numericas   = [c for c in X_train.columns if c not in categoricas]

# ========================================
# 3️⃣ OneHot transformer
# ========================================
onehot = ColumnTransformer(
    [("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=True), categoricas)],
    remainder="passthrough"
)

# ========================================
# 4️⃣ Pipeline XGBoost
# ========================================
pipeline_xgb = Pipeline(steps=[
    ("onehot", onehot),
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        tree_method="hist",
        use_label_encoder=False,
        random_state=42
    ))
])

# ========================================
# 5️⃣ Espacio HPO
# ========================================
from scipy.stats import randint, uniform

param_dist = {
    "xgb__max_depth": randint(3, 8),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.6, 0.4),
    "xgb__colsample_bytree": uniform(0.6, 0.4),
    "xgb__gamma": uniform(0, 5),
    "xgb__min_child_weight": randint(1, 10),
    "xgb__n_estimators": randint(200, 800)
}

# ========================================
# 6️⃣ Cross-validation
# ========================================
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# ========================================
# 7️⃣ RandomizedSearchCV para PRECISIÓN
# ========================================
search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=15,          # menos iteraciones para acelerar
    scoring="precision",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

# ========================================
# 8️⃣ Entrenar
# ========================================
search_xgb.fit(X_train, y_train)

# ========================================
# 9️⃣ Resultados HPO
# ========================================
print("🏆 Mejores hiperparámetros para maximizar PRECISIÓN:")
print(search_xgb.best_params_)

# ========================================
# 🔟 Evaluar sobre test
# ========================================
y_pred_prob = search_xgb.predict_proba(X_test)[:,1]

# Métricas
auc = roc_auc_score(y_test, y_pred_prob)
gini = 2*auc - 1

# KS Score
def ks_score(y_true, y_score):
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df = df.sort_values("score", ascending=False)
    df["cum_event"] = (df["y"]==1).cumsum() / df["y"].sum()
    df["cum_nonevent"] = (df["y"]==0).cumsum() / (len(df) - df["y"].sum())
    ks = (df["cum_event"] - df["cum_nonevent"]).max()
    return ks

ks = ks_score(y_test, y_pred_prob)

print(f"\n📊 Métricas en TEST:")
print(f"AUC  : {auc:.4f}")
print(f"Gini : {gini:.4f}")
print(f"KS   : {ks:.4f}")


Fitting 3 folds for each of 15 candidates, totalling 45 fits


/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the 'manylinux2014' variant of XGBoost. Certain features such as GPU algorithms or federated learning are not available. To use these features, please upgrade to a recent Linux distro with glibc 2.28+, and install the 'manylinux_2_28' variant.
  warnings.warn(
/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/xgboost/core.py:377: FutureWarning: Your system has an old version of glibc (< 2.28). We will stop supporting Linux distros with glibc older than 2.28 after **May 31, 2025**. Please upgrade to a recent Linux distro (with glibc >= 2.28) to use future versions of XGBoost.
Note: You have installed the

🏆 Mejores hiperparámetros para maximizar PRECISIÓN:
{'xgb__colsample_bytree': 0.9329770563201687, 'xgb__gamma': 1.0616955533913808, 'xgb__learning_rate': 0.03727374508106509, 'xgb__max_depth': 7, 'xgb__min_child_weight': 1, 'xgb__n_estimators': 659, 'xgb__subsample': 0.8446612641953124}

📊 Métricas en TEST:
AUC  : 0.8723
Gini : 0.7446
KS   : 0.6327


In [90]:
# ==============================================================
# 7) Generar deciles y micro-segmentación (CORREGIDO)
# ==============================================================

# Asegurar que df_test tenga los scores del modelo
df_eval = df_test.copy()
df_eval['score'] = y_pred_prob  # score del mejor modelo


# ==============================================================
# 7a) DECILES (10 grupos)
# ==============================================================
df_eval["decile"] = pd.qcut(df_eval["score"].rank(method="first"), 10, labels=False) + 1

tabla_deciles = df_eval.groupby("decile").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean")
).reset_index()

tabla_deciles["event_rate"] = tabla_deciles["events"] / tabla_deciles["count"]
tabla_deciles["precision"] = tabla_deciles["event_rate"]

# ORDEN CORRECTO: mayor score → menor score
tabla_deciles = tabla_deciles.sort_values("score_mean", ascending=False).reset_index(drop=True)

# RECALL ACUMULADO
tabla_deciles["recall"] = tabla_deciles["events"].cumsum() / tabla_deciles["events"].sum()

tabla_deciles["lift"] = tabla_deciles["event_rate"] / df_eval["target"].mean()

print("\n📊 TABLA POR DECILES:")
print(tabla_deciles)


📊 TABLA POR DECILES:
   decile  count  events  score_mean  event_rate  precision  recall  lift
0      10     58      44        0.90        0.76       0.76    0.35  3.42
1       9     57      29        0.53        0.51       0.51    0.57  2.30
2       8     57      26        0.15        0.46       0.46    0.78  2.06
3       7     57      12        0.04        0.21       0.21    0.87  0.95
4       6     57       5        0.02        0.09       0.09    0.91  0.40
5       5     58       4        0.01        0.07       0.07    0.94  0.31
6       4     57       2        0.00        0.04       0.04    0.96  0.16
7       3     57       2        0.00        0.04       0.04    0.98  0.16
8       2     57       1        0.00        0.02       0.02    0.98  0.08
9       1     58       2        0.00        0.03       0.03    1.00  0.16


In [91]:
# ==============================================================
# 7) Generar QUINTILES con todas las métricas que pediste
# ==============================================================
df_eval = df_test.copy()
df_eval['score'] = y_pred_prob  # probabilidad del modelo (clase positiva)

# Crear quintiles basados en el score (qcut divide por cuantiles del score)
df_eval["quintil"] = pd.qcut(df_eval["score"], q=5, labels=False) + 1

# Invertir para que quintil 1 = mayor riesgo más alto
df_eval["quintil"] = 6 - df_eval["quintil"]

# Tabla agrupada
tabla_quintiles = (
    df_eval.groupby("quintil")
    .agg(
        cantidad_casos=("target", "size"),
        casos_positivos=("target", "sum"),
        prob_min=("score", "min"),
        prob_max=("score", "max")
    )
    .reset_index()
)

# Precision
tabla_quintiles["precision"] = tabla_quintiles["casos_positivos"] / tabla_quintiles["cantidad_casos"]

# Rango de probabilidad como texto
tabla_quintiles["rango_probabilidad"] = (
    tabla_quintiles["prob_min"].round(4).astype(str) + " - " + 
    tabla_quintiles["prob_max"].round(4).astype(str)
)

# Ordenar de mayor a menor riesgo (quintil 1 al 5)
tabla_quintiles = tabla_quintiles.sort_values("quintil", ascending=True).reset_index(drop=True)

# Recall acumulado (¡CORREGIDO!)
total_positivos = tabla_quintiles["casos_positivos"].sum()
tabla_quintiles["recall"] = tabla_quintiles["casos_positivos"].cumsum() / total_positivos

# Opcional: lift
tabla_quintiles["lift"] = tabla_quintiles["precision"] / df_eval["target"].mean()

# Seleccionar y ordenar columnas finales
tabla_quintiles = tabla_quintiles[[
    "quintil",
    "cantidad_casos",
    "casos_positivos",
    "rango_probabilidad",
    "precision",
    "recall"
]]

# Redondear para mejor presentación
tabla_quintiles["precision"] = tabla_quintiles["precision"].round(4)
tabla_quintiles["recall"]    = tabla_quintiles["recall"].round(4)

print("\nTABLA POR QUINTILES (Quintil 1 = mayor riesgo):")
print(tabla_quintiles.to_string(index=False))


TABLA POR QUINTILES (Quintil 1 = mayor riesgo):
 quintil  cantidad_casos  casos_positivos rango_probabilidad  precision  recall
       1             115               73    0.2862 - 0.9881       0.63    0.57
       2             114               38    0.0252 - 0.2836       0.33    0.87
       3             115                9     0.0047 - 0.025       0.08    0.94
       4             114                4    0.0011 - 0.0046       0.04    0.98
       5             115                3     1e-04 - 0.0011       0.03    1.00


In [92]:
# ==============================================================
# 7) Generar deciles y micro-segmentación (CORREGIDO)
# ==============================================================

# Asegurar que df_test tenga los scores del modelo
df_eval = df_test.copy()
df_eval['score'] = y_pred_prob  # score del mejor modelo


# ==============================================================
# 7a) DECILES (10 grupos)
# ==============================================================
df_eval["decile"] = pd.qcut(df_eval["score"].rank(method="first"), 5, labels=False) + 1

tabla_deciles = df_eval.groupby("decile").agg(
    count=("target", "size"),
    events=("target", "sum"),
    score_mean=("score", "mean")
).reset_index()

tabla_deciles["event_rate"] = tabla_deciles["events"] / tabla_deciles["count"]
tabla_deciles["precision"] = tabla_deciles["event_rate"]

# ORDEN CORRECTO: mayor score → menor score
tabla_deciles = tabla_deciles.sort_values("score_mean", ascending=False).reset_index(drop=True)

# RECALL ACUMULADO
tabla_deciles["recall"] = tabla_deciles["events"].cumsum() / tabla_deciles["events"].sum()

tabla_deciles["lift"] = tabla_deciles["event_rate"] / df_eval["target"].mean()

print("\n📊 TABLA POR DECILES:")
print(tabla_deciles)


📊 TABLA POR DECILES:
   decile  count  events  score_mean  event_rate  precision  recall  lift
0       5    115      73        0.72        0.63       0.63    0.57  2.86
1       4    114      38        0.10        0.33       0.33    0.87  1.50
2       3    115       9        0.01        0.08       0.08    0.94  0.35
3       2    114       4        0.00        0.04       0.04    0.98  0.16
4       1    115       3        0.00        0.03       0.03    1.00  0.12
[CV] END xgb__colsample_bytree=0.749816047538945, xgb__gamma=4.75357153204958, xgb__learning_rate=0.11979909127171076, xgb__max_depth=7, xgb__min_child_weight=5, xgb__n_estimators=321, xgb__subsample=0.662397808134481; total time=   0.8s
[CV] END xgb__colsample_bytree=0.6232334448672797, xgb__gamma=4.330880728874676, xgb__learning_rate=0.10016725176148131, xgb__max_depth=5, xgb__min_child_weight=6, xgb__n_estimators=508, xgb__subsample=0.9879639408647978; total time=   0.9s
[CV] END xgb__colsample_bytree=0.6028265220878869, xgb_

In [102]:
#df_all = df_5.copy()   # O el df con preprocess completo (label encoding)

In [93]:
def tabla_quintiles_por_mes(df_mes, modelo, target_col="target"):
    """
    df_mes: dataframe filtrado por mes
    modelo: search_xgb (RandomizedSearchCV entrenado)
    """
    if df_mes.empty:
        return None
    
    X = df_mes.drop(columns=[target_col])
    y = df_mes[target_col]

    # Score
    score = modelo.predict_proba(X)[:, 1]

    df_temp = df_mes.copy()
    df_temp["score"] = score

    # Quintiles (5 grupos)
    df_temp["quintil"] = pd.qcut(
        df_temp["score"].rank(method="first"), 
        5, 
        labels=False
    ) + 1

    tabla = df_temp.groupby("quintil").agg(
        count=(target_col, "size"),
        events=(target_col, "sum"),
        score_mean=("score", "mean")
    ).reset_index()

    tabla["event_rate"] = tabla["events"] / tabla["count"]
    tabla["precision"] = tabla["event_rate"]

    tabla = tabla.sort_values("score_mean", ascending=False).reset_index(drop=True)

    # Recall acumulado
    tabla["recall"] = tabla["events"].cumsum() / tabla["events"].sum()

    # Lift
    tabla["lift"] = tabla["event_rate"] / df_temp[target_col].mean()

    return tabla


In [94]:
meses = [202506, 202507, 202508, 202509]

salidas = {}

for m in meses:
    df_mes = df_all[df_all["mes_base"] == m].copy()
    tabla = tabla_quintiles_por_mes(df_mes, search_xgb, target_col="target")
    salidas[m] = tabla


In [95]:
for m in salidas:
    print(f"\n📌 Resultados para el mes {m}")
    display(salidas[m])



📌 Resultados para el mes 202506


,quintil,count,events,score_mean,event_rate,precision,recall,lift
0,5,508,115,0.22,0.23,0.23,0.99,4.96
1,4,508,0,0.01,0.00,0.00,0.99,0.00
2,3,508,1,0.00,0.00,0.00,1.00,0.04
3,2,508,0,0.00,0.00,0.00,1.00,0.00
4,1,509,0,0.00,0.00,0.00,1.00,0.00



📌 Resultados para el mes 202507


,quintil,count,events,score_mean,event_rate,precision,recall,lift
0,5,496,103,0.20,0.21,0.21,1.00,5.00
1,4,496,0,0.01,0.00,0.00,1.00,0.00
2,3,496,0,0.00,0.00,0.00,1.00,0.00
3,2,496,0,0.00,0.00,0.00,1.00,0.00
4,1,497,0,0.00,0.00,0.00,1.00,0.00



📌 Resultados para el mes 202508


,quintil,count,events,score_mean,event_rate,precision,recall,lift
0,5,115,73,0.72,0.63,0.63,0.57,2.86
1,4,114,38,0.10,0.33,0.33,0.87,1.50
2,3,115,9,0.01,0.08,0.08,0.94,0.35
3,2,114,4,0.00,0.04,0.04,0.98,0.16
4,1,115,3,0.00,0.03,0.03,1.00,0.12



📌 Resultados para el mes 202509


None

In [96]:
list(df_4)

['target',
 'num_documento',
 'mes_base',
 'pasivo_soles',
 'trx_monto_abonos_6m_efectivo',
 'trx_monto_cargos_6m_efectivo',
 'trx_q_cargos_3m_total',
 'trx_q_abonos_promedio_3m_total',
 'trx_monto_cargos_promedio_3m_total',
 'trx_q_abonos_ratio_1m_efectivo_total',
 'trx_q_abonos_ratio_3m_efectivo_total',
 'trx_q_abonos_ratio_9m_efectivo_total',
 'trx_monto_cargos_ratio_1m_efectivo_total',
 'trx_monto_abonos_3m_max',
 'edad_constitucion',
 'antiguedad',
 'nivel_riesgo_lsb_total',
 'q_meses_ingresos_0',
 'q_meses_egresos_0',
 'provincia',
 'departamento',
 'ubigeo_cd',
 'sectorista_id',
 'ciiu_v4',
 'flag_casos_hist',
 'flag_variacion_abono_monto_total_5m_1m',
 'flag_variacion_efect_cargos_monto_5m_1m',
 'q_ro_debajo_umbral',
 'q_alerta_hist',
 'q_ros_hist',
 'tipo_alerta_n2']

In [97]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) Asegurarnos de tener el score en df_test + variables originales
# ==============================================================
# Suponiendo que df_4 es tu dataframe original completo (con columnas originales)
# y que ya tienes: y_pred_prob = search_xgb.predict_proba(X_test)[:,1]
df_final = df_4[df_4["mes_base"] == 202508].copy()  # solo período de evaluación
df_final["score"] = y_pred_prob

# Seleccionar solo las variables relevantes del modelo
variables_modelo = [
    "trx_monto_abonos_3m_max", "sectorista_id", "pasivo_soles", "q_meses_ingresos_0",
    "ciiu_v4", "trx_monto_abonos_6m_efectivo", "q_meses_egresos_0", "antiguedad",
    "trx_monto_cargos_6m_efectivo",
    "trx_q_abonos_promedio_3m_total", "trx_q_abonos_ratio_9m_efectivo_total",
    "edad_constitucion", "ubigeo_cd", "trx_q_abonos_ratio_1m_efectivo_total",
    "trx_monto_cargos_ratio_1m_efectivo_total", "departamento",
    "trx_monto_cargos_promedio_3m_total","tipo_alerta_n2"
]

# Asegurarse de que todas las columnas existan (ignorar si alguna falta)
df_final = df_final[variables_modelo + ["score", "target", "num_documento"]].copy()

# ==============================================================
# 2) Crear quintiles (5 grupos) por score descendente
# ==============================================================
df_final["quintil"] = pd.qcut(df_final["score"], 5, labels=["Q1","Q2","Q3","Q4","Q5"][::-1])
# Q1 = más riesgoso (top 20%), Q5 = menos riesgoso

# ==============================================================
# 3) Calcular métricas por quintil usando las variables del modelo
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("num_documento", "nunique"),
    monto_abonos_3m_max=("trx_monto_abonos_3m_max", "mean"),  # Máximo abono 3m
    pasivo_soles_prom=("pasivo_soles", "mean"),  # Promedio pasivos en soles
    q_meses_ingresos_0_prom=("q_meses_ingresos_0", "mean"),  # Promedio meses sin ingresos
    abonos_6m_efectivo_prom=("trx_monto_abonos_6m_efectivo", "mean"),  # Promedio abonos en efectivo 6m
    q_meses_egresos_0_prom=("q_meses_egresos_0", "mean"),  # Promedio meses sin egresos
    antiguedad_prom=("antiguedad", "mean"),  # Promedio años de relación
  #  dif_abonos_cargos_6m_prom=("dif_q_abonos_cargos_efectivo_6m", "mean"),  # Dif. abonos-cargos 6m
    cargos_6m_efectivo_prom=("trx_monto_cargos_6m_efectivo", "mean"),  # Promedio cargos en efectivo 6m
    abonos_3m_total_prom=("trx_q_abonos_promedio_3m_total", "mean"),  # Promedio abonos totales 3m
    ratio_abonos_9m_efectivo=("trx_q_abonos_ratio_9m_efectivo_total", "mean"),  # Ratio abonos 9m
    edad_constitucion_prom=("edad_constitucion", "mean"),  # Promedio años de constitución
    riesgo_real_pct=("target", "mean"),  # % de clientes con evento real
    riesgo_real_abs=("target", "sum"),
    pct_lima=("departamento", lambda x: (x=="LIMA").mean() * 100),
    pct_trujillo=("departamento", lambda x: (x=="LA LIBERTAD").mean() * 100),
    pct_san_roman=("departamento", lambda x: (x=="PUNO").mean() * 100),
    # Alertas con riesgo (basado en tipo_alerta_n2, asumimos que está en df_final)
    pct_alertas_riesgo=("tipo_alerta_n2", lambda x: (x.str.contains("ALTO|Riesgo", case=False, na=False)).mean() * 100)
).round(2)

# Ordenar de mayor a menor riesgo
resumen = resumen.loc[["Q1", "Q2", "Q3", "Q4", "Q5"]].copy()
resumen["clientes_acum"] = resumen["clientes"].cumsum()
total_clientes = resumen["clientes"].sum()
total_riesgo = resumen["riesgo_real_abs"].sum()

resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["riesgo_real_abs"].cumsum() / total_riesgo * 100).round(1)

# ==============================================================
# 4) Imprimir los valores para el HTML
# ==============================================================
print("VALORES PARA EL HTML (cópialos directamente):")
print("="*60)

s1 = resumen.loc["Q1"]
s2 = resumen.loc["Q2"]
s3 = resumen.loc["Q3"]
s4_s5 = resumen.loc[["Q4", "Q5"]].sum()

# Formatear montos con formato peruano (S/)
def format_soles(valor):
    return f"S/ {valor:,.1f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"S1_clientes = {int(s1.clientes)}")
print(f"S1_monto_abonos_3m_max = {format_soles(s1.monto_abonos_3m_max)}")
print(f"S1_pasivo_soles = {format_soles(s1.pasivo_soles_prom)}")
print(f"S1_q_meses_ingresos_0 = {s1.q_meses_ingresos_0_prom:.1f}")
print(f"S1_abonos_6m_efectivo = {format_soles(s1.abonos_6m_efectivo_prom)}")
print(f"S1_q_meses_egresos_0 = {s1.q_meses_egresos_0_prom:.1f}")
print(f"S1_antiguedad = {s1.antiguedad_prom:.1f}")

print(f"S1_cargos_6m_efectivo = {format_soles(s1.cargos_6m_efectivo_prom)}")
print(f"S1_abonos_3m_total = {s1.abonos_3m_total_prom:.1f}")
print(f"S1_ratio_abonos_9m = {s1.ratio_abonos_9m_efectivo:.2f}")
print(f"S1_edad_constitucion = {s1.edad_constitucion_prom:.1f}")
print(f"S1_riesgo_real = {s1.riesgo_real_pct:.2f}")
print(f"S1_lima = {s1.pct_lima:.1f}")
print(f"S1_trujillo = {s1.pct_trujillo:.1f}")
print(f"S1_san_roman = {s1.pct_san_roman:.1f}")
print(f"S1_alertas_riesgo = {s1.pct_alertas_riesgo:.1f}")

print(f"\nS2_clientes = {int(s2.clientes)}")
print(f"S2_monto_abonos_3m_max = {format_soles(s2.monto_abonos_3m_max)}")
print(f"S2_pasivo_soles = {format_soles(s2.pasivo_soles_prom)}")
print(f"S2_q_meses_ingresos_0 = {s2.q_meses_ingresos_0_prom:.1f}")
print(f"S2_abonos_6m_efectivo = {format_soles(s2.abonos_6m_efectivo_prom)}")
print(f"S2_q_meses_egresos_0 = {s2.q_meses_egresos_0_prom:.1f}")
print(f"S2_antiguedad = {s2.antiguedad_prom:.1f}")

print(f"S2_cargos_6m_efectivo = {format_soles(s2.cargos_6m_efectivo_prom)}")
print(f"S2_abonos_3m_total = {s2.abonos_3m_total_prom:.1f}")
print(f"S2_ratio_abonos_9m = {s2.ratio_abonos_9m_efectivo:.2f}")
print(f"S2_edad_constitucion = {s2.edad_constitucion_prom:.1f}")
print(f"S2_lima = {s2.pct_lima:.1f}")
print(f"S2_trujillo = {s2.pct_trujillo:.1f}")
print(f"S2_san_roman = {s2.pct_san_roman:.1f}")
print(f"S2_alertas_riesgo = {s2.pct_alertas_riesgo:.1f}")

print(f"\nS3_clientes = {int(s3.clientes)}")
print(f"S3_monto_abonos_3m_max = {format_soles(s3.monto_abonos_3m_max)}")
print(f"S3_pasivo_soles = {format_soles(s3.pasivo_soles_prom)}")
print(f"S3_q_meses_ingresos_0 = {s3.q_meses_ingresos_0_prom:.1f}")
print(f"S3_abonos_6m_efectivo = {format_soles(s3.abonos_6m_efectivo_prom)}")
print(f"S3_q_meses_egresos_0 = {s3.q_meses_egresos_0_prom:.1f}")
print(f"S3_antiguedad = {s3.antiguedad_prom:.1f}")

print(f"S3_cargos_6m_efectivo = {format_soles(s3.cargos_6m_efectivo_prom)}")
print(f"S3_abonos_3m_total = {s3.abonos_3m_total_prom:.1f}")
print(f"S3_ratio_abonos_9m = {s3.ratio_abonos_9m_efectivo:.2f}")
print(f"S3_edad_constitucion = {s3.edad_constitucion_prom:.1f}")

print(f"\nS4S5_clientes = {int(s4_s5.clientes)}")
print(f"S4S5_monto_abonos_3m_max = {format_soles(s4_s5.monto_abonos_3m_max)}")
print(f"S4S5_pasivo_soles = {format_soles(s4_s5.pasivo_soles_prom)}")
print(f"S4S5_q_meses_ingresos_0 = {s4_s5.q_meses_ingresos_0_prom:.1f}")
print(f"S4S5_abonos_6m_efectivo = {format_soles(s4_s5.abonos_6m_efectivo_prom)}")
print(f"S4S5_q_meses_egresos_0 = {s4_s5.q_meses_egresos_0_prom:.1f}")
print(f"S4S5_antiguedad = {s4_s5.antiguedad_prom:.1f}")

print(f"S4S5_cargos_6m_efectivo = {format_soles(s4_s5.cargos_6m_efectivo_prom)}")
print(f"S4S5_abonos_3m_total = {s4_s5.abonos_3m_total_prom:.1f}")
print(f"S4S5_ratio_abonos_9m = {s4_s5.ratio_abonos_9m_efectivo:.2f}")
print(f"S4S5_edad_constitucion = {s4_s5.edad_constitucion_prom:.1f}")

print(f"\nCONCENTRACIÓN:")
print(f"pct_s1_s2 = {resumen.loc[['Q1','Q2'],'pct_clientes'].sum():.1f}")
print(f"pct_riesgo_s1_s2 = {resumen.loc[['Q1','Q2'],'pct_riesgo_acum'].iloc[-1]:.1f}")
print(f"pct_alertas_s1_s2 = {(resumen.loc[['Q1','Q2'],'pct_alertas_riesgo'].mean()):.1f}")

VALORES PARA EL HTML (cópialos directamente):
S1_clientes = 109
S1_monto_abonos_3m_max = S/ 3.526.618,9
S1_pasivo_soles = S/ 188.617,1
S1_q_meses_ingresos_0 = 6.8
S1_abonos_6m_efectivo = S/ 1.172.868,7
S1_q_meses_egresos_0 = 6.0
S1_antiguedad = 0.8
S1_cargos_6m_efectivo = S/ 1.022.371,0
S1_abonos_3m_total = 39.3
S1_ratio_abonos_9m = 0.46
S1_edad_constitucion = 1.2
S1_riesgo_real = 0.63
S1_lima = 40.9
S1_trujillo = 13.9
S1_san_roman = 16.5
S1_alertas_riesgo = 0.0

S2_clientes = 110
S2_monto_abonos_3m_max = S/ 3.745.166,8
S2_pasivo_soles = S/ 1.083.736,2
S2_q_meses_ingresos_0 = 3.7
S2_abonos_6m_efectivo = S/ 1.531.184,0
S2_q_meses_egresos_0 = 3.3
S2_antiguedad = 2.6
S2_cargos_6m_efectivo = S/ 798.340,6
S2_abonos_3m_total = 52.8
S2_ratio_abonos_9m = 0.27
S2_edad_constitucion = 6.2
S2_lima = 56.1
S2_trujillo = 9.7
S2_san_roman = 7.9
S2_alertas_riesgo = 0.0

S3_clientes = 115
S3_monto_abonos_3m_max = S/ 3.935.155,4
S3_pasivo_soles = S/ 659.634,4
S3_q_meses_ingresos_0 = 2.4
S3_abonos_6m_efec

In [113]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) QUINTILES CORRECTOS: S1 = MÁS RIESGO (score más alto)
# ==============================================================
# Forzamos 5 grupos exactamente iguales en cantidad de clientes
df_final["quintil"] = pd.qcut(-df_final["score"], 
                              q=5, 
                              labels=["S1", "S2", "S3", "S4", "S5"])

# ==============================================================
# 2) RESUMEN GENERAL
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("num_documento", "nunique"),
    eventos=("target", "sum"),
    event_rate=("target", "mean"),
    score_prom=("score", "mean"),
    
    abonos_3m_max_prom=("trx_monto_abonos_3m_max", "mean"),
    pasivo_soles_prom=("pasivo_soles", "mean"),
    q_meses_sin_ingresos=("q_meses_ingresos_0", "mean"),
    antiguedad_prom=("antiguedad", "mean"),
    edad_constitucion_prom=("edad_constitucion", "mean"),
    
    pct_lima=("departamento", lambda x: (x == "LIMA").mean() * 100),
    pct_trujillo=("departamento", lambda x: (x == "LA LIBERTAD").mean() * 100),
).round(3)

# Métricas globales
total_clientes = resumen["clientes"].sum()
total_eventos = resumen["eventos"].sum()

resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["eventos"].cumsum() / total_eventos * 100).round(1)
tasa_global = df_final["target"].mean()
resumen["lift"] = (resumen["event_rate"] / tasa_global).round(2)

# ==============================================================
# 3) TIPOS DE ALERTA
# ==============================================================
if "tipo_alerta_n2" not in df_final.columns:
    df_final = df_final.merge(df_4[["num_documento", "mes_base", "tipo_alerta_n2"]],
                              on=["num_documento", "mes_base"], how="left")

df_final["alerta"] = df_final["tipo_alerta_n2"].fillna("SIN ALERTA").str.strip().str.upper()

alertas = (df_final.groupby("quintil")["alerta"]
           .value_counts(normalize=True)
           .mul(100)
           .round(1)
           .unstack(fill_value=0))

# ==============================================================
# 4) FUNCIÓN PARA FORMATO S/
# ==============================================================
def soles(x):
    return f"S/ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

# ==============================================================
# 5) IMPRESIÓN BONITA Y LISTA PARA LA DIAPOSITIVA
# ==============================================================
print("\n" + "="*100)
print("RESULTADOS POR QUINTIL – PLAFT 2025 – LISTO PARA PRESENTACIÓN")
print("="*100)

for seg in ["S1", "S2", "S3"]:
    s = resumen.loc[seg]
    print(f"\n{seg} → {int(s.clientes)} clientes | {int(s.eventos)} casos reales | {s.event_rate*100:.1f}% event rate | Lift {s.lift}x")
    print(f"   Score promedio: {s.score_prom:.4f}")
    print(f"   Abonos máx 3m: {soles(s.abonos_3m_max_prom)}")
    print(f"   Pasivos: {soles(s.pasivo_soles_prom)}")
    print(f"   Antigüedad: {s.antiguedad_prom:.1f} años | Edad constitución: {s.edad_constitucion_prom:.1f} años")
    print(f"   Meses sin ingresos: {s.q_meses_sin_ingresos:.1f}")
    print(f"   % Lima: {s.pct_lima:.1f}% | % Trujillo: {s.pct_trujillo:.1f}%")
    print(f"   Alertas → ", end="")
    print(" | ".join([f"{col}: {alertas.loc[seg,col]}%" for col in alertas.columns if alertas.loc[seg,col] > 0]))

# S4+S5 juntos
s4s5 = resumen.loc[["S4","S5"]].sum()
s4s5.event_rate = s4s5.eventos / s4s5.clientes
s4s5.lift = round(s4s5.event_rate / tasa_global, 2)

print(f"\nS4+S5 → {int(s4s5.clientes)} clientes | {int(s4s5.eventos)} casos reales | {s4s5.event_rate*100:.1f}% event rate | Lift {s4s5.lift}x")
print(f"   Casi todo el volumen actual de alertas está aquí… pero con riesgo mínimo")

print(f"\nCONCENTRACIÓN FINAL:")
print(f"• S1+S2 = {resumen.loc[['S1','S2'],'pct_clientes'].sum():.1f}% de los clientes")
print(f"• Capturan el {resumen.loc[['S1','S2'],'pct_riesgo_acum'].iloc[-1]:.1f}% del riesgo real")
print(f"• Lift promedio S1+S2 ≈ {resumen.loc[['S1','S2'],'lift'].mean():.2f}x")

print("\n" + "="*100)
print("¡PEGA ESTA SALIDA EN EL CHAT Y EN 2 MINUTOS TE DOY EL HTML FINAL MÁS BRUTAL QUE HAYAS TENIDO!")
print("="*100)


RESULTADOS POR QUINTIL – PLAFT 2025 – LISTO PARA PRESENTACIÓN

S1 → 107 clientes | 79 casos reales | 68.7% event rate | Lift 3.1x
   Score promedio: 0.7770
   Abonos máx 3m: S/ 4.224.989
   Pasivos: S/ 281.744
   Antigüedad: 0.7 años | Edad constitución: 0.9 años
   Meses sin ingresos: 6.8
   % Lima: 43.5% | % Trujillo: 13.0%
   Alertas → AUTOMATICA: 33.9% | MANUAL: 46.1% | SEMI AUTOMATICA: 20.0%

S2 → 112 clientes | 30 casos reales | 26.3% event rate | Lift 1.19x
   Score promedio: 0.0970
   Abonos máx 3m: S/ 3.362.893
   Pasivos: S/ 1.168.848
   Antigüedad: 2.6 años | Edad constitución: 7.1 años
   Meses sin ingresos: 4.2
   % Lima: 51.8% | % Trujillo: 10.5%
   Alertas → AUTOMATICA: 66.7% | MANUAL: 21.1% | SEMI AUTOMATICA: 12.3%

S3 → 115 clientes | 13 casos reales | 11.3% event rate | Lift 0.51x
   Score promedio: 0.0100
   Abonos máx 3m: S/ 4.777.637
   Pasivos: S/ 829.478
   Antigüedad: 7.4 años | Edad constitución: 8.7 años
   Meses sin ingresos: 1.7
   % Lima: 60.0% | % Trujill

In [116]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) CREAR QUINTILES CORRECTOS: S1 = MÁS ALTO SCORE = MÁS RIESGO
# ==============================================================
df_final["quintil"] = pd.qcut(-df_final["score"], 5, labels=["S1", "S2", "S3", "S4", "S5"])

# ==============================================================
# 2) TODAS LAS VARIABLES DEL MODELO (promedios)
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("num_documento", "nunique"),
    eventos=("target", "sum"),
    event_rate=("target", "mean"),
    score_prom=("score", "mean"),
    
    # Variables del modelo (promedios)
    abonos_3m_max_prom=("trx_monto_abonos_3m_max", "mean"),
    pasivo_soles_prom=("pasivo_soles", "mean"),
    q_meses_sin_ingresos=("q_meses_ingresos_0", "mean"),
    abonos_6m_efectivo_prom=("trx_monto_abonos_6m_efectivo", "mean"),
    q_meses_sin_egresos=("q_meses_egresos_0", "mean"),
    antiguedad_prom=("antiguedad", "mean"),
    cargos_6m_efectivo_prom=("trx_monto_cargos_6m_efectivo", "mean"),
    abonos_promedio_3m_total=("trx_q_abonos_promedio_3m_total", "mean"),
    ratio_abonos_9m_efectivo=("trx_q_abonos_ratio_9m_efectivo_total", "mean"),
    edad_constitucion_prom=("edad_constitucion", "mean"),
    ratio_abonos_1m_efectivo=("trx_q_abonos_ratio_1m_efectivo_total", "mean"),
    ratio_cargos_1m_efectivo=("trx_monto_cargos_ratio_1m_efectivo_total", "mean"),
    cargos_promedio_3m_total=("trx_monto_cargos_promedio_3m_total", "mean"),
    
    # Geografía
    pct_lima=("departamento", lambda x: (x == "LIMA").mean() * 100),
    pct_trujillo=("departamento", lambda x: (x == "LA LIBERTAD").mean() * 100),
    pct_san_roman=("departamento", lambda x: (x == "PUNO").mean() * 100)
).round(3)

# Métricas globales
total_clientes = resumen["clientes"].sum()
total_eventos = resumen["eventos"].sum()
resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["eventos"].cumsum() / total_eventos * 100).round(1)
resumen["lift"] = (resumen["event_rate"] / df_final["target"].mean()).round(2)

# ==============================================================
# 3) TIPOS DE ALERTA POR SEGMENTO
# ==============================================================
if "tipo_alerta_n2" not in df_final.columns:
    df_final = df_final.merge(df_4[["num_documento", "mes_base", "tipo_alerta_n2"]], 
                              on=["num_documento", "mes_base"], how="left")

df_final["alerta"] = df_final["tipo_alerta_n2"].fillna("SIN ALERTA").str.strip().str.upper()

alertas = (df_final.groupby("quintil")["alerta"]
           .value_counts(normalize=True)
           .mul(100)
           .round(1)
           .unstack(fill_value=0))

# ==============================================================
# 4) IMPRIMIR TODO (S1, S2, S3, S4+S5) - LISTO PARA HTML
# ==============================================================
print("\n" + "="*90)
print("RESULTADOS COMPLETOS POR SEGMENTO - LISTO PARA DIAPOSITIVA PLAFT")
print("="*90)

def soles(x):
    return f"S/ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

seg_names = ["S1", "S2", "S3"]
for seg in seg_names + ["S4+S5"]:
    if seg != "S4+S5":
        s = resumen.loc[seg]
        print(f"\n--- {seg} (Top riesgo) ---")
    else:
        s = resumen.loc[["S4","S5"]].sum()
        s.event_rate = (s.eventos / s.clientes)
        s.lift = (s.event_rate / df_final["target"].mean()).round(2)
        print(f"\n--- {seg} (Bajo riesgo) ---")
    
    print(f"Clientes: {int(s.clientes)} | Eventos: {int(s.eventos)} | Event Rate: {s.event_rate*100:.1f}% | Lift: {s.lift:.2f}x")
    print(f"Score promedio: {s.score_prom:.4f}")
    print(f"Abonos máx 3m (prom): {soles(s.abonos_3m_max_prom)}")
    print(f"Pasivos soles (prom): {soles(s.pasivo_soles_prom)}")
    print(f"Antigüedad (prom): {s.antiguedad_prom:.1f} años")
 
    print(f"Cargos efectivo 6m (prom): {soles(s.abonos_6m_efectivo_prom)}")
    print(f"Meses sin ingresos (prom): {s.q_meses_sin_ingresos:.1f}")
    print(f"Ratio abonos efectivo 9m: {s.ratio_abonos_9m_efectivo:.3f}")
    print(f"Edad constitución: {s.edad_constitucion_prom:.1f} años")
    print(f"% Lima: {s.pct_lima:.1f}%")
    
    # Alertas
    if seg != "S4+S5":
        print(f"TIPOS DE ALERTA {seg}:")
        for col in alertas.columns:
            if seg in alertas.index and alertas.loc[seg, col] > 0:
                print(f"   → {col}: {alertas.loc[seg, col]}%")
    else:
        print(f"TIPOS DE ALERTA S4+S5:")
        s4s5_alert = df_final[df_final["quintil"].isin(["S4","S5"])]["alerta"].value_counts(normalize=True).mul(100).round(1)
        for tipo, pct in s4s5_alert.items():
            print(f"   → {tipo}: {pct}%")

print(f"\nCONCENTRACIÓN FINAL:")
print(f"S1+S2 clientes: {resumen.loc[['S1','S2'],'pct_clientes'].sum():.1f}%")
print(f"S1+S2 riesgo capturado: {resumen.loc[['S1','S2'],'pct_riesgo_acum'].iloc[-1]:.1f}%")

print("\n" + "="*90)
print("¡EJECUTA Y PÉGAME TODA ESTA SALIDA PARA ARMAR EL HTML DEFINITIVO!")
print("="*90)


RESULTADOS COMPLETOS POR SEGMENTO - LISTO PARA DIAPOSITIVA PLAFT

--- S1 (Top riesgo) ---
Clientes: 107 | Eventos: 79 | Event Rate: 68.7% | Lift: 3.10x
Score promedio: 0.7770
Abonos máx 3m (prom): S/ 4.224.989
Pasivos soles (prom): S/ 281.744
Antigüedad (prom): 0.7 años
Cargos efectivo 6m (prom): S/ 1.244.912
Meses sin ingresos (prom): 6.8
Ratio abonos efectivo 9m: 0.484
Edad constitución: 0.9 años
% Lima: 43.5%
TIPOS DE ALERTA S1:
   → AUTOMATICA: 33.9%
   → MANUAL: 46.1%
   → SEMI AUTOMATICA: 20.0%

--- S2 (Top riesgo) ---
Clientes: 112 | Eventos: 30 | Event Rate: 26.3% | Lift: 1.19x
Score promedio: 0.0970
Abonos máx 3m (prom): S/ 3.362.893
Pasivos soles (prom): S/ 1.168.848
Antigüedad (prom): 2.6 años
Cargos efectivo 6m (prom): S/ 808.893
Meses sin ingresos (prom): 4.2
Ratio abonos efectivo 9m: 0.214
Edad constitución: 7.1 años
% Lima: 51.8%
TIPOS DE ALERTA S2:
   → AUTOMATICA: 66.7%
   → MANUAL: 21.1%
   → SEMI AUTOMATICA: 12.3%

--- S3 (Top riesgo) ---
Clientes: 115 | Eventos: 13

In [ ]:
import pandas as pd
import numpy as np

# ==============================================================
# 1) CREAR QUINTILES CORRECTOS: S1 = MÁS ALTO SCORE = MÁS RIESGO
# ==============================================================
# Esta es la línea mágica que arregla todo en tu caso:
df_final["quintil"] = pd.qcut(-df_final["score"], 5, labels=["S1", "S2", "S3", "S4", "S5"])
# El signo "-" invierte el orden → los scores más altos van a S1

# ==============================================================
# 2) MÉTRICAS CLAVE POR SEGMENTO
# ==============================================================
resumen = df_final.groupby("quintil").agg(
    clientes=("num_documento", "nunique"),
    eventos=("target", "sum"),
    event_rate=("target", "mean"),
    score_prom=("score", "mean"),
    monto_max_abono_3m=("trx_monto_abonos_3m_max", "mean"),
    pasivo_soles=("pasivo_soles", "mean"),
    antiguedad=("antiguedad", "mean"),
    dif_abonos_cargos=("dif_q_abonos_cargos_efectivo_6m", "mean"),
    cargos_efectivo_6m=("trx_monto_cargos_6m_efectivo", "mean"),
    pct_lima=("departamento", lambda x: (x == "LIMA").mean() * 100)
).round(3)

total_clientes = resumen["clientes"].sum()
total_eventos = resumen["eventos"].sum()
resumen["pct_clientes"] = (resumen["clientes"] / total_clientes * 100).round(1)
resumen["pct_riesgo_acum"] = (resumen["eventos"].cumsum() / total_eventos * 100).round(1)
resumen["lift"] = (resumen["event_rate"] / df_final["target"].mean()).round(2)

# ==============================================================
# 3) TIPOS DE ALERTA POR SEGMENTO (AUTOMÁTICA, MANUAL, etc.)
# ==============================================================
if "tipo_alerta_n2" not in df_final.columns:
    df_final = df_final.merge(
        df_4[["num_documento", "mes_base", "tipo_alerta_n2"]],
        on=["num_documento", "mes_base"],
        how="left"
    )

df_final["alerta"] = df_final["tipo_alerta_n2"].fillna("SIN ALERTA").str.strip().str.upper()

alertas = (
    df_final.groupby("quintil")["alerta"]
    .value_counts(normalize=True)
    .mul(100)
    .round(1)
    .unstack(fill_value=0)
)

# ==============================================================
# 4) IMPRIMIR TODO LISTO PARA EL HTML
# ==============================================================
print("\n" + "="*80)
print("SALIDA FINAL PARA TU DIAPOSITIVA PLAFT (S1 = MÁS RIESGO)")
print("="*80)

s1 = resumen.loc["S1"]
s2 = resumen.loc["S2"]
s3 = resumen.loc["S3"]
s4s5 = resumen.loc[["S4", "S5"]].sum()

def soles(x):
    return f"S/ {x:,.0f}".replace(",", "X").replace(".", ",").replace("X", ".")

print(f"S1_clientes = {int(s1.clientes)}")
print(f"S1_eventos = {int(s1.eventos)}")
print(f"S1_event_rate = {s1.event_rate*100:.1f}%")
print(f"S1_lift = {s1.lift:.2f}x")
print(f"S1_monto_max_3m = {soles(s1.monto_max_abono_3m)}")
print(f"S1_pasivo_prom = {soles(s1.pasivo_soles)}")
print(f"S1_antiguedad = {s1.antiguedad:.1f} años")
print(f"S1_dif_abonos_cargos = {s1.dif_abonos_cargos:+.1f}")
print(f"S1_pct_lima = {s1.pct_lima:.1f}%")

print(f"\nS2_clientes = {int(s2.clientes)}")
print(f"S2_eventos = {int(s2.eventos)}")
print(f"S2_event_rate = {s2.event_rate*100:.1f}%")
print(f"S2_lift = {s2.lift:.2f}x")

print(f"\nCONCENTRACIÓN S1+S2:")
print(f"Clientes_S1S2 = {int(s1.clientes + s2.clientes)} ({resumen.loc[['S1','S2'],'pct_clientes'].sum():.1f}%)")
print(f"Riesgo_capturado_S1S2 = {resumen.loc[['S1','S2'],'pct_riesgo_acum'].iloc[-1]:.1f}%")

print(f"\nTIPOS DE ALERTA POR SEGMENTO")
print("-"*60)
for seg, nombre in [("S1","S1"), ("S2","S2"), ("S3","S3")]:
    print(f"\n→ {nombre}:")
    if seg in alertas.index:
        for tipo in alertas.columns:
            pct = alertas.loc[seg, tipo]
            if pct > 0:
                print(f"   • {tipo}: {pct}%")
    else:
        print("   • Sin datos")

print(f"\n→ S4+S5:")
alertas_s4s5 = df_final[df_final["quintil"].isin(["S4","S5"])]["alerta"].value_counts(normalize=True).mul(100).round(1)
for tipo, pct in alertas_s4s5.items():
    print(f"   • {tipo}: {pct}%")

print("\n" + "="*80)
print("¡EJECUTA ESTO Y PÉGAME LA SALIDA COMPLETA!")
print("="*80)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, precision_score, recall_score

# -------------------------------
# 1️⃣ Matriz con umbral 0.3
# -------------------------------
umbral_default = 0.35
y_pred_default = (y_pred_prob >= umbral_default).astype(int)

# Matriz de confusión
cm_default = confusion_matrix(y_test, y_pred_default)
disp_default = ConfusionMatrixDisplay(confusion_matrix=cm_default)
print(f"\n📌 Matriz de Confusión (umbral {umbral_default})")
disp_default.plot()

# -------------------------------
# 2️⃣ Precision y Recall
# -------------------------------
precision = precision_score(y_test, y_pred_default)
recall = recall_score(y_test, y_pred_default)

print(f"\n🎯 Precision: {precision:.4f}")
print(f"📈 Recall: {recall:.4f}")

# Base completa con 2 meses test y 2 meses de val

In [ ]:
import pandas as pd
from sklearn.utils import resample

# ============================================================
# 1️⃣ DEFINIR SPLIT DE DATOS
# ============================================================

df_train_raw = df_3[df_3["mes_base"] < 202506].copy()
df_val_raw   = df_3[df_3["mes_base"].isin([202506, 202507])].copy()
df_test_raw  = df_3[df_3["mes_base"].isin([202508, 202509])].copy()


# ============================================================
# 2️⃣ EN TRAIN: separar con alerta y sin alerta
# ============================================================

df_train_con_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] != "0"]
df_train_sin_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] == "0"]


# ============================================================
# 3️⃣ Separar clases en TRAIN
# ============================================================

df_train_1 = df_train_con_alerta[df_train_con_alerta["target"] == 1]
df_train_0 = df_train_con_alerta[df_train_con_alerta["target"] == 0]


# ============================================================
# 4️⃣ Balanceo:
#        Queremos 0.5% → target=1
#        Queremos 99.5% → target=0
# ============================================================

n_pos = len(df_train_1)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # relación 199:1

if n_neg_deseado <= len(df_train_0):
    df_train_0_bal = resample(df_train_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_train_0_bal = resample(df_train_0, replace=True, n_samples=n_neg_deseado, random_state=42)

df_train_1_bal = df_train_1


# ============================================================
# 5️⃣ Agregar 0.5% de casos SIN alerta
# ============================================================

df_sin_alerta_sample = df_train_sin_alerta.sample(frac=0.005, random_state=42)


# ============================================================
# 6️⃣ Construir TRAIN final
# ============================================================

df_train = pd.concat([
    df_train_0_bal,
    df_train_1_bal,
    df_sin_alerta_sample
], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)


# ============================================================
# 7️⃣ VALIDACIÓN → se deja intacta
# ============================================================

df_val = df_val_raw.copy()


# ============================================================
# 8️⃣ TEST → meses 202508 y 202509 completos
# ============================================================

df_test = df_test_raw.copy()


# ============================================================
# 9️⃣ SALIDAS
# ============================================================

print("📌 TRAIN (target):")
print(df_train["target"].value_counts(normalize=True))

print("\n📌 VAL (target):")
print(df_val["target"].value_counts(normalize=True))

print("\n📌 TEST (target):")
print(df_test["target"].value_counts(normalize=True))

print("\nTamaños finales:")
print("TRAIN:", df_train.shape)
print("VAL:  ", df_val.shape)
print("TEST: ", df_test.shape)


In [ ]:
df_5 = pd.concat([df_train,df_val, df_test], axis=0).reset_index(drop=True)

In [ ]:
cols_a_eliminar = ["fecha_constitucion"]
df_5 = df_5.drop(columns=cols_a_eliminar, errors="ignore")
df_5[['mes_base', 'target']].value_counts()
# Convertir explícitamente la columna problemática a string
df_5['tipo_alerta_n2'] = df_5['tipo_alerta_n2'].astype(str)

In [ ]:
df_5[['mes_base', 'target']].value_counts()

In [ ]:
df_5.to_parquet(
    's3://ibk-discovery-comercial-us-east-1-654654352211-data/discovery/comercial/sanherna/PLAFT/PJ/MINORISTA/DATA_INFERENCIA/data_pn_total_train_val_test.parquet',
    index=False
)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Separar TRAIN / VAL / TEST por mes_base
# ============================================================



# Guardamos para reconstruir después
df_all = pd.concat([df_train, df_val, df_test], axis=0).copy()

from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Codificar categorías en df_all (NO hacemos splits aún)
# ============================================================

df_encoded = df_all.copy()

categoricas = [
    "flag_casos_hist",
    "flag_variacion_abono_monto_total_5m_1m",
    "flag_variacion_efect_cargos_monto_5m_1m",
    "provincia",
    "departamento",
    "ubigeo_cd",
    "sectorista_id",
    "ciiu_v4"
]

encoders = {}

for c in categoricas:
    le = LabelEncoder()
    df_encoded[c] = le.fit_transform(df_encoded[c].astype(str))
    encoders[c] = le

# ============================================================
# 2️⃣ Ahora sí reconstruimos los splits con mes_base original
# ============================================================

df_train = df_encoded[df_encoded["mes_base"] <= 202505].copy()       # Train
df_val   = df_encoded[df_encoded["mes_base"].isin([202506, 202507])].copy()  # Val
df_test  = df_encoded[df_encoded["mes_base"].isin([202508, 202509])].copy()  # Test

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)



In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform

# =====================================================
# 1️⃣ Separar features / target
# =====================================================

target = "target"

X_train = df_train.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_train = df_train[target]

X_val   = df_val.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_val   = df_val[target]

X_test  = df_test.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_test  = df_test[target]

# =====================================================
# 2️⃣ Todas las variables son numéricas (LabelEncoded)
# =====================================================

pipeline_xgb = Pipeline(steps=[
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    ))
])

# =====================================================
# 3️⃣ HPO
# =====================================================

param_dist = {
    "xgb__max_depth": randint(3, 8),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.6, 0.4),
    "xgb__colsample_bytree": uniform(0.6, 0.4),
    "xgb__gamma": uniform(0, 5),
    "xgb__min_child_weight": randint(1, 10),
    "xgb__n_estimators": randint(200, 800)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="precision",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search_xgb.fit(X_train, y_train)

print("\n🏆 Mejores hiperparámetros:")
print(search_xgb.best_params_)

# =====================================================
# 4️⃣ Evaluación en TEST
# =====================================================

y_pred_prob = search_xgb.predict_proba(X_test)[:,1]

def ks_score(y_true, y_score):
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df = df.sort_values("score", ascending=False)
    df["cum_event"] = (df["y"]==1).cumsum() / df["y"].sum()
    df["cum_nonevent"] = (df["y"]==0).cumsum() / (len(df) - df["y"].sum())
    return (df["cum_event"] - df["cum_nonevent"]).max()

auc  = roc_auc_score(y_test, y_pred_prob)
gini = 2*auc - 1
ks   = ks_score(y_test, y_pred_prob)

print("\n📊 MÉTRICAS EN TEST:")
print(f"AUC   = {auc:.4f}")
print(f"Gini  = {gini:.4f}")
print(f"KS    = {ks:.4f}")


In [ ]:
import pandas as pd
import numpy as np

df_val["score"]  = search_xgb.predict_proba(X_val)[:,1]
df_test["score"] = search_xgb.predict_proba(X_test)[:,1]

df_eval = pd.concat([df_val, df_test], axis=0).reset_index(drop=True)

print("Meses incluidos:", df_eval["mes_base"].unique())

# ======================================================================
# 1️⃣ Definir percentiles acumulativos
# ======================================================================
percentiles = [0.01, 0.05, 0.10, 0.50, 1.00]
labels = ["Top 1%", "Top 5%", "Top 10%", "Top 50%", "Total"]

resultados = []

# ======================================================================
# 2️⃣ Loop por mes
# ======================================================================
for mes in sorted(df_eval["mes_base"].unique()):
    
    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    
    # Ordenar descendentemente por score
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)
    
    # Ranking en percentil
    df_mes["percentile"] = (df_mes.index + 1) / len(df_mes)

    tablas = []
    
    for p, label in zip(percentiles, labels):
        df_cut = df_mes[df_mes["percentile"] <= p]

        tabla = {
            "grupo": label,
            "count": len(df_cut),
            "events": df_cut["target"].sum(),
            "precision": df_cut["target"].mean(),
            "recall": df_cut["target"].sum() / df_mes["target"].sum(),
            "score_mean": df_cut["score"].mean(),
            "mes_base": mes
        }
        tablas.append(tabla)
    
    resultados.extend(tablas)

# ======================================================================
# 3️⃣ Resultado final
# ======================================================================
df_result_final = pd.DataFrame(resultados)

print("\n📊 EFECTIVIDAD ACUMULADA POR GRUPOS (1%,5%,10%,50%,100%)")
print(df_result_final)



In [ ]:
import pandas as pd

# ===============================================================
# CONFIG: cantidad de grupos
# ===============================================================
N_GROUPS = 500      # 500 grupos iguales
TOP_SHOW = 5        # mostrar solo los primeros 5

resultados = []

for mes in sorted(df_eval["mes_base"].unique()):

    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)

    # Crear grupos 1..500 (ordenados por score)
    df_mes["grupo"] = (df_mes.index // (len(df_mes) / N_GROUPS)).astype(int) + 1
    df_mes.loc[df_mes["grupo"] > N_GROUPS, "grupo"] = N_GROUPS

    # Resumen por grupo
    tabla = df_mes.groupby("grupo").agg(
        count=("target", "size"),
        events=("target", "sum"),
        precision=("target", "mean"),
        score_mean=("score", "mean")
    ).reset_index()

    # Ordenados de mejor a peor
    tabla = tabla.sort_values("score_mean", ascending=False).reset_index(drop=True)

    # Recall acumulado
    total_eventos_mes = tabla["events"].sum()
    tabla["recall_acum"] = tabla["events"].cumsum() / total_eventos_mes

    # Lift
    tasa_base = df_mes["target"].mean()
    tabla["lift"] = tabla["precision"] / tasa_base

    # Mantener solo los TOP 5 del mes
    tabla = tabla.head(TOP_SHOW)
    tabla["mes_base"] = mes

    resultados.append(tabla)

# Unir todo
df_top_grupos = pd.concat(resultados, axis=0, ignore_index=True)

# Mostrar
print("\n📊 EFECTIVIDAD — TOP 5 DE 500 GRUPOS POR MES (Precision, Recall, Lift)")
print(df_top_grupos)


In [ ]:
#solo variables del modelo de Camila no estaN ['trx_q_abonos_6m_total', 'trx_q_abonos_6m_efectivo', 'trx_q_cargos_6m_total']

In [ ]:
camila = [
    'nivel_riesgo_lsb_ultima',
    'flag_desv_activa',
    'flag_ros_hist',
    'flag_casos_hist',
    'cp_cantidad_ing',
    'trx_monto_abonos_6m_total',
    'trx_monto_abonos_ratio_6m_efectivo_total',
    'q_meses_ingresos_0',
    'pasivo_soles',
    'edad_constitucion',
    'flag_variacion_abono_monto_total_5m_1m',
    'flag_variacion_efect_cargos_monto_5m_1m',
    'antiguedad',
    'dif_monto_abonos_cargos_efectivo_6m',
    'dif_q_abonos_cargos_efectivo_6m','mes_base', 'tipo_alerta_n2', 'num_documento','target_m'
]


In [ ]:
df_4=df_dataset[camila]

In [ ]:
import pandas as pd
from sklearn.utils import resample

# ============================================================
# 1️⃣ DEFINIR SPLIT DE DATOS
# ============================================================

df_train_raw = df_4[df_4["mes_base"] < 202506].copy()
df_val_raw   = df_4[df_4["mes_base"].isin([202506, 202507])].copy()
df_test_raw  = df_4[df_4["mes_base"].isin([202508, 202509])].copy()


# ============================================================
# 2️⃣ EN TRAIN: separar con alerta y sin alerta
# ============================================================

df_train_con_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] != "0"]
df_train_sin_alerta  = df_train_raw[df_train_raw["tipo_alerta_n2"] == "0"]


# ============================================================
# 3️⃣ Separar clases en TRAIN
# ============================================================

df_train_1 = df_train_con_alerta[df_train_con_alerta["target_m"] == 1]
df_train_0 = df_train_con_alerta[df_train_con_alerta["target_m"] == 0]


# ============================================================
# 4️⃣ Balanceo:
#        Queremos 0.5% → target=1
#        Queremos 99.5% → target=0
# ============================================================

n_pos = len(df_train_1)
n_neg_deseado = int(n_pos * (99.5 / 0.5))  # relación 199:1

if n_neg_deseado <= len(df_train_0):
    df_train_0_bal = resample(df_train_0, replace=False, n_samples=n_neg_deseado, random_state=42)
else:
    df_train_0_bal = resample(df_train_0, replace=True, n_samples=n_neg_deseado, random_state=42)

df_train_1_bal = df_train_1


# ============================================================
# 5️⃣ Agregar 0.5% de casos SIN alerta
# ============================================================

df_sin_alerta_sample = df_train_sin_alerta.sample(frac=0.005, random_state=42)


# ============================================================
# 6️⃣ Construir TRAIN final
# ============================================================

df_train = pd.concat([
    df_train_0_bal,
    df_train_1_bal,
    df_sin_alerta_sample
], axis=0).sample(frac=1, random_state=42).reset_index(drop=True)


# ============================================================
# 7️⃣ VALIDACIÓN → se deja intacta
# ============================================================

df_val = df_val_raw.copy()


# ============================================================
# 8️⃣ TEST → meses 202508 y 202509 completos
# ============================================================

df_test = df_test_raw.copy()


# ============================================================
# 9️⃣ SALIDAS
# ============================================================

print("📌 TRAIN (target):")
print(df_train["target_m"].value_counts(normalize=True))

print("\n📌 VAL (target):")
print(df_val["target_m"].value_counts(normalize=True))

print("\n📌 TEST (target):")
print(df_test["target_m"].value_counts(normalize=True))

print("\nTamaños finales:")
print("TRAIN:", df_train.shape)
print("VAL:  ", df_val.shape)
print("TEST: ", df_test.shape)

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Separar TRAIN / VAL / TEST por mes_base
# ============================================================



# Guardamos para reconstruir después
df_all = pd.concat([df_train, df_val, df_test], axis=0).copy()

from sklearn.preprocessing import LabelEncoder
import pandas as pd

# ============================================================
# 1️⃣ Codificar categorías en df_all (NO hacemos splits aún)
# ============================================================

df_encoded = df_all.copy()

categoricas = [
    "flag_casos_hist"
]

encoders = {}

for c in categoricas:
    le = LabelEncoder()
    df_encoded[c] = le.fit_transform(df_encoded[c].astype(str))
    encoders[c] = le

# ============================================================
# 2️⃣ Ahora sí reconstruimos los splits con mes_base original
# ============================================================

df_train = df_encoded[df_encoded["mes_base"] <= 202505].copy()       # Train
df_val   = df_encoded[df_encoded["mes_base"].isin([202506, 202507])].copy()  # Val
df_test  = df_encoded[df_encoded["mes_base"].isin([202508, 202509])].copy()  # Test

print("Train:", df_train.shape)
print("Val:  ", df_val.shape)
print("Test: ", df_test.shape)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score
from scipy.stats import randint, uniform

# =====================================================
# 1️⃣ Separar features / target
# =====================================================

target = "target_m"

X_train = df_train.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_train = df_train[target]

X_val   = df_val.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_val   = df_val[target]

X_test  = df_test.drop(columns=[target, "mes_base", "tipo_alerta_n2", "num_documento"], errors="ignore")
y_test  = df_test[target]

# =====================================================
# 2️⃣ Todas las variables son numéricas (LabelEncoded)
# =====================================================

pipeline_xgb = Pipeline(steps=[
    ("xgb", XGBClassifier(
        eval_metric="logloss",
        tree_method="hist",
        random_state=42
    ))
])

# =====================================================
# 3️⃣ HPO
# =====================================================

param_dist = {
    "xgb__max_depth": randint(3, 8),
    "xgb__learning_rate": uniform(0.01, 0.15),
    "xgb__subsample": uniform(0.6, 0.4),
    "xgb__colsample_bytree": uniform(0.6, 0.4),
    "xgb__gamma": uniform(0, 5),
    "xgb__min_child_weight": randint(1, 10),
    "xgb__n_estimators": randint(200, 800)
}

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_dist,
    n_iter=20,
    scoring="precision",
    cv=cv,
    verbose=2,
    n_jobs=-1,
    random_state=42
)

search_xgb.fit(X_train, y_train)

print("\n🏆 Mejores hiperparámetros:")
print(search_xgb.best_params_)

# =====================================================
# 4️⃣ Evaluación en TEST
# =====================================================

y_pred_prob = search_xgb.predict_proba(X_test)[:,1]

def ks_score(y_true, y_score):
    df = pd.DataFrame({"y": y_true, "score": y_score})
    df = df.sort_values("score", ascending=False)
    df["cum_event"] = (df["y"]==1).cumsum() / df["y"].sum()
    df["cum_nonevent"] = (df["y"]==0).cumsum() / (len(df) - df["y"].sum())
    return (df["cum_event"] - df["cum_nonevent"]).max()

auc  = roc_auc_score(y_test, y_pred_prob)
gini = 2*auc - 1
ks   = ks_score(y_test, y_pred_prob)

print("\n📊 MÉTRICAS EN TEST:")
print(f"AUC   = {auc:.4f}")
print(f"Gini  = {gini:.4f}")
print(f"KS    = {ks:.4f}")

In [ ]:
# ===============================================================
# 1️⃣ Agregar score
# ===============================================================
df_val["score"]  = search_xgb.predict_proba(X_val)[:,1]
df_test["score"] = search_xgb.predict_proba(X_test)[:,1]

df_eval = pd.concat([df_val, df_test], axis=0).reset_index(drop=True)

# ===============================================================
# 2️⃣ Construir 1000 grupos por mes y mostrar los 10 MEJORES
# ===============================================================

N_GRUPOS = 500
resultados = []

for mes in sorted(df_eval["mes_base"].unique()):

    df_mes = df_eval[df_eval["mes_base"] == mes].copy()

    # Crear grupos de score (1000)
    df_mes["grupo_1000"] = pd.qcut(
        df_mes["score"].rank(method="first"),
        q=N_GRUPOS,
        labels=False
    ) + 1

    tabla = df_mes.groupby("grupo_1000").agg(
        count=("target_m", "size"),
        events=("target_m", "sum"),
        score_mean=("score", "mean")
    ).reset_index()

    tabla["precision"] = tabla["events"] / tabla["count"]

    total_eventos_mes = df_mes["target_m"].sum()
    tabla["recall_acum"] = tabla["events"].cumsum() / total_eventos_mes

    tabla["mes_base"] = mes

    # ❗ AHORA SÍ: mejores 10 → score más ALTO
    tabla = tabla.sort_values("score_mean", ascending=False).head(10)

    resultados.append(tabla)

# ===============================================================
# 3️⃣ Resultado final
# ===============================================================

df_top10_1000grupos = pd.concat(resultados, ignore_index=True)
df_top10_1000grupos = df_top10_1000grupos.sort_values(["mes_base", "score_mean"], ascending=[True, False])

print(df_top10_1000grupos)

In [ ]:
import pandas as pd

# ===============================================================
# CONFIG: cantidad de grupos
# ===============================================================
N_GROUPS = 500      # 500 grupos iguales
TOP_SHOW = 5        # mostrar solo los primeros 5

resultados = []

for mes in sorted(df_eval["mes_base"].unique()):

    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)

    # Crear grupos 1..500 (ordenados por score)
    df_mes["grupo"] = (df_mes.index // (len(df_mes) / N_GROUPS)).astype(int) + 1
    df_mes.loc[df_mes["grupo"] > N_GROUPS, "grupo"] = N_GROUPS

    # Resumen por grupo
    tabla = df_mes.groupby("grupo").agg(
        count=("target_m", "size"),
        events=("target_m", "sum"),
        precision=("target_m", "mean"),
        score_mean=("score", "mean")
    ).reset_index()

    # Ordenados de mejor a peor
    tabla = tabla.sort_values("score_mean", ascending=False).reset_index(drop=True)

    # Recall acumulado
    total_eventos_mes = tabla["events"].sum()
    tabla["recall_acum"] = tabla["events"].cumsum() / total_eventos_mes

    # Lift
    tasa_base = df_mes["target_m"].mean()
    tabla["lift"] = tabla["precision"] / tasa_base

    # Mantener solo los TOP 5 del mes
    tabla = tabla.head(TOP_SHOW)
    tabla["mes_base"] = mes

    resultados.append(tabla)

# Unir todo
df_top_grupos = pd.concat(resultados, axis=0, ignore_index=True)

# Mostrar
print("\n📊 EFECTIVIDAD — TOP 5 DE 500 GRUPOS POR MES (Precision, Recall, Lift)")
print(df_top_grupos)


In [ ]:
import pandas as pd
import numpy as np

df_val["score"]  = search_xgb.predict_proba(X_val)[:,1]
df_test["score"] = search_xgb.predict_proba(X_test)[:,1]

df_eval = pd.concat([df_val, df_test], axis=0).reset_index(drop=True)

print("Meses incluidos:", df_eval["mes_base"].unique())

# ======================================================================
# 1️⃣ Definir percentiles acumulativos
# ======================================================================
percentiles = [0.01, 0.05, 0.10, 0.50, 1.00]
labels = ["Top 1%", "Top 5%", "Top 10%", "Top 50%", "Total"]

resultados = []

# ======================================================================
# 2️⃣ Loop por mes
# ======================================================================
for mes in sorted(df_eval["mes_base"].unique()):
    
    df_mes = df_eval[df_eval["mes_base"] == mes].copy()
    
    # Ordenar descendentemente por score
    df_mes = df_mes.sort_values("score", ascending=False).reset_index(drop=True)
    
    # Ranking en percentil
    df_mes["percentile"] = (df_mes.index + 1) / len(df_mes)

    tablas = []
    
    for p, label in zip(percentiles, labels):
        df_cut = df_mes[df_mes["percentile"] <= p]

        tabla = {
            "grupo": label,
            "count": len(df_cut),
            "events": df_cut["target_m"].sum(),
            "precision": df_cut["target_m"].mean(),
            "recall": df_cut["target_m"].sum() / df_mes["target_m"].sum(),
            "score_mean": df_cut["score"].mean(),
            "mes_base": mes
        }
        tablas.append(tabla)
    
    resultados.extend(tablas)

# ======================================================================
# 3️⃣ Resultado final
# ======================================================================
df_result_final = pd.DataFrame(resultados)

print("\n📊 EFECTIVIDAD ACUMULADA POR GRUPOS (1%,5%,10%,50%,100%)")
print(df_result_final)
